In [2]:
# CELL: Ingest Accela Zoning Reports into a separate database
print("📊 INGESTING ACCELA ZONING REPORTS\n")
print("="*70)

import pandas as pd
import sqlite3
import glob
import os
from datetime import datetime

report_dir = '/Users/johngage/berkeley-data/zoning_reports'
DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

# List all xlsx files
files = sorted(glob.glob(f'{report_dir}/*.xlsx'))
print(f"Found {len(files)} Excel files in {report_dir}:\n")
for f in files:
    size = os.path.getsize(f) / 1024
    mtime = datetime.fromtimestamp(os.path.getmtime(f)).strftime('%Y-%m-%d')
    print(f"   📄 {os.path.basename(f)} ({size:.0f} KB, {mtime})")

# Read the three March 19 files
print(f"\n{'='*70}")
print("\nReading today's reports:\n")

target_files = {
    'active_landuse_v1': '2026_03_19_ActiveLandUse_V1.xlsx',
    'active_landuse_v1_2': '2026_03_19_ActiveLandUse_V1_2.xlsx',
    'active_landuse_v1_all': '2026_03_19_ActiveLandUse_V1_all.xlsx',
}

conn = sqlite3.connect(DB_PATH)

for table_name, filename in target_files.items():
    filepath = os.path.join(report_dir, filename)
    if os.path.exists(filepath):
        try:
            xl = pd.ExcelFile(filepath)
            print(f"\n📁 {filename}")
            print(f"   Sheets: {xl.sheet_names}")
            
            for sheet in xl.sheet_names:
                df = pd.read_excel(filepath, sheet_name=sheet)
                print(f"\n   Sheet '{sheet}': {len(df)} rows, {len(df.columns)} columns")
                print(f"   Columns: {df.columns.tolist()}")
                
                if len(df) > 0:
                    print(f"   Sample:")
                    display(df.head(3))
                
                safe_name = f"{table_name}_{sheet}".replace(' ', '_').replace('-', '_')
                df.to_sql(safe_name, conn, if_exists='replace', index=False)
                print(f"   💾 Saved as '{safe_name}'")
                
        except Exception as e:
            print(f"   ❌ Error reading {filename}: {e}")
    else:
        print(f"   ⚠️ File not found: {filepath}")

# Show all tables
print(f"\n{'='*70}")
print(f"\n📊 All tables in {os.path.basename(DB_PATH)}:")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
for t in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM [{t}]", conn).iloc[0]['n']
    print(f"   📋 {t}: {count:,} rows")

conn.close()
print(f"\n💾 Database: {DB_PATH}")

📊 INGESTING ACCELA ZONING REPORTS

Found 9 Excel files in /Users/johngage/berkeley-data/zoning_reports:

   📄 2026-1-14ActiveLandUse_V1.xlsx (20 KB, 2026-01-14)
   📄 2026-1-6_ActiveLandUse_V1.xlsx (20 KB, 2026-01-06)
   📄 2026_03_19_ActiveLandUse_V1.xlsx (20 KB, 2026-03-19)
   📄 2026_03_19_ActiveLandUse_V1_2.xlsx (17 KB, 2026-03-19)
   📄 2026_03_19_ActiveLandUse_V1_all.xlsx (21 KB, 2026-03-19)
   📄 6_LandUseStatus_V1.xlsx (8 KB, 2025-12-08)
   📄 ActiveLandUse_V1_All_permits.xlsx (20 KB, 2025-12-08)
   📄 LandUseStatus_V1_OpenAppeal.xlsx (8 KB, 2025-12-08)
   📄 Zoning_active_projects_ActiveLandUse_V1.xlsx (21 KB, 2025-12-08)


Reading today's reports:


📁 2026_03_19_ActiveLandUse_V1.xlsx
   Sheets: ['ActiveLandUse_V1']

   Sheet 'ActiveLandUse_V1': 154 rows, 4 columns
   Columns: ['Land Use Status\nAs of 3/19/2026 4:25 PM', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3']
   Sample:


,Land Use Status\nAs of 3/19/2026 4:25 PM,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,NaN,NaN,NaN,NaN
1,Permit Number,Description,Address,Record Status
2,ZP2017-0081,Use Permit to add 3 wireless antennas to an ex...,1804 EUCLID Ave,Pending


   💾 Saved as 'active_landuse_v1_ActiveLandUse_V1'

📁 2026_03_19_ActiveLandUse_V1_2.xlsx
   Sheets: ['ActiveLandUse_V1']

   Sheet 'ActiveLandUse_V1': 135 rows, 4 columns
   Columns: ['Land Use Status\nAs of 3/19/2026 4:51 PM', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3']
   Sample:


,Land Use Status\nAs of 3/19/2026 4:51 PM,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,NaN,NaN,NaN,NaN
1,Permit Number,Description,Address,Record Status
2,PLN2022-0111,Misc. Pre-Application meeting for a proposed n...,1099 ASHBY Ave,Pending


   💾 Saved as 'active_landuse_v1_2_ActiveLandUse_V1'

📁 2026_03_19_ActiveLandUse_V1_all.xlsx
   Sheets: ['ActiveLandUse_V1']

   Sheet 'ActiveLandUse_V1': 155 rows, 4 columns
   Columns: ['Land Use Status\nAs of 3/19/2026 4:58 PM', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3']
   Sample:


,Land Use Status\nAs of 3/19/2026 4:58 PM,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,NaN,NaN,NaN,NaN
1,Permit Number,Description,Address,Record Status
2,ZP2017-0081,Use Permit to add 3 wireless antennas to an ex...,1804 EUCLID Ave,Pending


   💾 Saved as 'active_landuse_v1_all_ActiveLandUse_V1'


📊 All tables in accela_reports.db:
   📋 active_landuse_v1_2_ActiveLandUse_V1: 135 rows
   📋 active_landuse_v1_ActiveLandUse_V1: 154 rows
   📋 active_landuse_v1_all_ActiveLandUse_V1: 155 rows

💾 Database: /Users/johngage/berkeley-data/databases/accela_reports.db


## found three spreadsheets in march 2026. Do they include all 2025 projects?

In [3]:
# CELL: Clean and properly parse the Accela reports
print("🔧 CLEANING ACCELA ZONING REPORTS\n")
print("="*70)

import pandas as pd
import sqlite3
import os
from datetime import datetime

report_dir = '/Users/johngage/berkeley-data/zoning_reports'
DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'

# The "all" file has the most records (155) — use that as the primary
filepath = os.path.join(report_dir, '2026_03_19_ActiveLandUse_V1_all.xlsx')

# Read with header on row 1 (skip the title row)
df_raw = pd.read_excel(filepath, header=None, skiprows=2)
df_raw.columns = ['Permit_Number', 'Description', 'Address', 'Record_Status']

# Drop empty rows
df = df_raw.dropna(subset=['Permit_Number']).copy()
df = df[df['Permit_Number'] != 'Permit Number']  # remove any repeated headers

print(f"✅ Cleaned: {len(df)} active zoning projects\n")
print(f"Columns: {df.columns.tolist()}")

# Show status distribution
print(f"\n📋 Record Status distribution:")
for status, count in df['Record_Status'].value_counts().items():
    print(f"   {status}: {count}")

# Show permit type distribution (extract prefix from permit number)
df['Permit_Type'] = df['Permit_Number'].str.extract(r'^([A-Z]+)')
print(f"\n📋 Permit type distribution:")
for ptype, count in df['Permit_Type'].value_counts().items():
    print(f"   {ptype}: {count}")

# Parse addresses to extract street number and name for matching
df['addr_upper'] = df['Address'].str.upper().str.strip()

# Extract street number
df['street_num'] = df['Address'].str.extract(r'^(\d+)').astype(float)

print(f"\n📋 Sample records:")
display(df.head(10))

# Save cleaned version
conn = sqlite3.connect(DB_PATH)
df.to_sql('active_zoning_projects', conn, if_exists='replace', index=False)
print(f"\n💾 Saved 'active_zoning_projects' ({len(df)} rows) to {os.path.basename(DB_PATH)}")

# Now try to match with parcels from berkeley.db
MAIN_DB = '/Users/johngage/berkeley-data/databases/berkeley.db'
if os.path.exists(MAIN_DB):
    conn_main = sqlite3.connect(MAIN_DB)
    
    # Load addresses from main db
    df_addr = pd.read_sql("SELECT apn_norm, FullAddress, OwnerName FROM addresses_arcgis", conn_main)
    df_addr['addr_upper'] = df_addr['FullAddress'].str.upper().str.strip()
    
    # Try matching on address
    df_matched = df.merge(
        df_addr,
        on='addr_upper',
        how='left',
        indicator=True
    )
    
    matched = (df_matched['_merge'] == 'both').sum()
    unmatched = (df_matched['_merge'] == 'left_only').sum()
    
    print(f"\n{'='*70}")
    print(f"\n🔗 MATCHING ZONING PROJECTS TO PARCELS:")
    print(f"   Matched by address:  {matched}")
    print(f"   Unmatched:           {unmatched}")
    print(f"   Match rate:          {matched/(matched+unmatched)*100:.1f}%")
    
    if unmatched > 0:
        print(f"\n   Unmatched addresses:")
        unmatched_df = df_matched[df_matched['_merge'] == 'left_only']
        for _, row in unmatched_df[['Permit_Number', 'Address', 'Record_Status']].head(10).iterrows():
            print(f"      {row['Permit_Number']}: {row['Address']} ({row['Record_Status']})")
    
    if matched > 0:
        print(f"\n   ✅ Sample matched projects with APN:")
        matched_df = df_matched[df_matched['_merge'] == 'both']
        for _, row in matched_df[['Permit_Number', 'Address', 'apn_norm', 'OwnerName', 'Record_Status']].head(10).iterrows():
            print(f"      {row['Permit_Number']}: {row['Address']}")
            print(f"         APN: {row['apn_norm']} | Owner: {row['OwnerName']} | Status: {row['Record_Status']}")
    
    # Save matched projects to main database
    if matched > 0:
        matched_df = df_matched[df_matched['_merge'] == 'both'].drop(columns=['_merge'])
        matched_df.to_sql('zoning_projects_with_parcels', conn_main, if_exists='replace', index=False)
        conn_main.execute('CREATE INDEX IF NOT EXISTS idx_zp_apn ON zoning_projects_with_parcels(apn_norm)')
        conn_main.commit()
        print(f"\n   💾 Saved 'zoning_projects_with_parcels' to berkeley.db")
    
    conn_main.close()

conn.close()
print(f"\n✅ Done!")

🔧 CLEANING ACCELA ZONING REPORTS

✅ Cleaned: 153 active zoning projects

Columns: ['Permit_Number', 'Description', 'Address', 'Record_Status']

📋 Record Status distribution:
   Incomplete Pending Applicant: 45
   In Review: 31
   Corrections Pending Applicant: 22
   Under Review: 20
   Pending Final Action: 20
   Pending: 7
   Approved: 2
   On Hold: 2
   Resubmittal Pending Review: 2
   Resubmittal Pending Staff: 2

📋 Permit type distribution:
   ZP: 153

📋 Sample records:


,Permit_Number,Description,Address,Record_Status,Permit_Type,addr_upper,street_num
1,ZP2017-0081,Use Permit to add 3 wireless antennas to an ex...,1804 EUCLID Ave,Pending,ZP,1804 EUCLID AVE,1804.0
2,ZP2017-0184,Legalize an existing carport on the front half...,1731 LA VEREDA Rd,In Review,ZP,1731 LA VEREDA RD,1731.0
3,ZP2018-0106,"597 SF addition above 14' to existing home, an...",1419 TENTH St,Incomplete Pending Applicant,ZP,1419 TENTH ST,1419.0
4,ZP2019-0088,to legalize an existing 8' tall fence within t...,2140 CURTIS St,Incomplete Pending Applicant,ZP,2140 CURTIS ST,2140.0
5,ZP2020-0006,New Single Family Dwelling on new parcel at 36...,40 HILL Rd,Incomplete Pending Applicant,ZP,40 HILL RD,40.0
6,ZP2020-0104,demolish an existing parking lot and portions ...,1914 FIFTH St,Under Review,ZP,1914 FIFTH ST,1914.0
7,ZP2021-0127,Demolish existing commercial building. Constru...,1710 UNIVERSITY Ave,Incomplete Pending Applicant,ZP,1710 UNIVERSITY AVE,1710.0
8,ZP2021-0155,Legalize a fence over 6 ft. in height within s...,624 VINCENTE Ave,Incomplete Pending Applicant,ZP,624 VINCENTE AVE,624.0
9,ZP2021-0158,50 unit multi-family mixed-use project,130 BERKELEY Sq,In Review,ZP,130 BERKELEY SQ,130.0
10,ZP2021-0215,Demolish existing commercial buildings and con...,1201 SECOND St,Under Review,ZP,1201 SECOND ST,1201.0



💾 Saved 'active_zoning_projects' (153 rows) to accela_reports.db


🔗 MATCHING ZONING PROJECTS TO PARCELS:
   Matched by address:  154
   Unmatched:           3
   Match rate:          98.1%

   Unmatched addresses:
      ZP2023-0063: 1850 BERRYMAN St (Incomplete Pending Applicant)
      ZP2023-0095: 2660 BANCROFT Way (Incomplete Pending Applicant)
      ZP2026-0002: 2163 UNIVERSITY Ave (Pending Final Action)

   ✅ Sample matched projects with APN:
      ZP2017-0081: 1804 EUCLID Ave
         APN: 058219100103 | Owner: HERCOWITZ MORIS & JANET TRS | Status: Pending
      ZP2017-0184: 1731 LA VEREDA Rd
         APN: 058220801200 | Owner: ZENTNER ROBERT P | Status: In Review
      ZP2018-0106: 1419 TENTH St
         APN: 059233202300 | Owner: REA COLLECTIVE LLC & NEWLINE PROPERTIES & INV ETAL | Status: Incomplete Pending Applicant
      ZP2019-0088: 2140 CURTIS St
         APN: 056198501400 | Owner: KOMOROUSKING VLASTA & ROBERT S | Status: Incomplete Pending Applicant
      ZP2020-0006: 40

In [4]:
# CELL: Verify berkeley.db is intact
import sqlite3, pandas as pd

conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type IN ('table','view') ORDER BY name", conn)
for t in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM [{t}]", conn).iloc[0]['n']
    print(f"   {'📋' if 'view' not in t.lower() else '👁️'} {t}: {count:,} rows")
conn.close()

   📋 addresses_arcgis: 65,459 rows
   📋 corridor_boundaries: 3 rows
   📋 corridor_far: 332 rows
   📋 corridor_master: 430 rows
   📋 corridor_ownership: 332 rows
   📋 development_potential: 41 rows
   📋 licenses: 13,004 rows
   📋 licenses_fts: 13,004 rows
   📋 licenses_fts_config: 1 rows
   📋 licenses_fts_data: 189 rows
   📋 licenses_fts_docsize: 12,970 rows
   📋 licenses_fts_idx: 170 rows
   📋 parcels: 29,024 rows
   📋 parcels_addresses_joined: 65,297 rows
   📋 parcels_arcgis: 29,024 rows
   📋 rent_control: 1,098 rows
   📋 zoning_districts: 42 rows
   📋 zoning_projects_with_parcels: 154 rows


In [5]:
# CELL: Analyze permit descriptions for housing production vs repair
print("🏠 ANALYZING PERMIT TYPES: Housing Production vs Repair\n")
print("="*70)

import pandas as pd
import sqlite3

DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql("SELECT * FROM active_zoning_projects", conn)
conn.close()

print(f"Total active zoning projects: {len(df)}\n")

# Show ALL descriptions so we can categorize
print("📋 ALL PROJECT DESCRIPTIONS:\n")
for _, row in df.iterrows():
    print(f"   {row['Permit_Number']:>15}: {row['Description']}")
    print(f"{'':>18} Status: {row['Record_Status']}")
    print()

🏠 ANALYZING PERMIT TYPES: Housing Production vs Repair

Total active zoning projects: 153

📋 ALL PROJECT DESCRIPTIONS:

       ZP2017-0081: Use Permit to add 3 wireless antennas to an existing facility with three existing antennas for a total of six antennas at 1804 Euclid previously approved by AUP; re-design and re-organize the antenna installation to include a new concealment screen
                   Status: Pending

       ZP2017-0184: Legalize an existing carport on the front half of a vacant lot and fence that exceeds 7 feet in height within a required setback.
                   Status: In Review

       ZP2018-0106: 597 SF addition above 14' to existing home, and new accessory building, as the fifth bedroom on site.
                   Status: Incomplete Pending Applicant

       ZP2019-0088: to legalize an existing 8' tall fence within the required side yard to the north and side yard to the east
                   Status: Incomplete Pending Applicant

       ZP2020-0006: New 

## Classifier

In [6]:
# CELL: Classify permits and extract unit counts
print("🏠 CLASSIFYING PERMITS & EXTRACTING UNIT COUNTS\n")
print("="*70)

import re
import pandas as pd
import sqlite3

DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM active_zoning_projects", conn)

def classify_and_extract(row):
    desc = str(row['Description']).lower() if pd.notna(row['Description']) else ''
    result = {
        'is_housing_production': False,
        'housing_type': 'Non-Housing',
        'estimated_units': 0,
        'has_density_bonus': False,
        'has_adu': False,
        'adds_bedrooms': False,
        'is_conversion': False,
        'is_sb330': False,
    }
    
    # Extract unit counts from description
    unit_patterns = [
        r'(\d+)\s*(?:dwelling|residential)\s*units?',
        r'(\d+)\s*units?',
        r'(\d+)-unit',
        r'(\d+)\s*(?:studio|bedroom)\s*units?',
        r'(\d+)\s*group living',
    ]
    
    max_units = 0
    for pattern in unit_patterns:
        matches = re.findall(pattern, desc)
        for m in matches:
            n = int(m)
            if n > max_units and n < 1000:  # sanity check
                max_units = n
    
    # Check density bonus
    result['has_density_bonus'] = 'density bonus' in desc
    result['is_sb330'] = 'sb-330' in desc or 'sb330' in desc or 'sb 330' in desc
    
    # ADU detection
    adu_keywords = ['adu', 'accessory dwelling', 'junior accessory', 'garage conversion to adu']
    result['has_adu'] = any(kw in desc for kw in adu_keywords)
    
    # Bedroom additions
    result['adds_bedrooms'] = 'bedroom' in desc and ('add' in desc or 'new' in desc or 'convert' in desc or 'create' in desc)
    
    # Conversion detection
    result['is_conversion'] = 'convert' in desc or 'conversion' in desc or 'change the use' in desc or 'change of use' in desc
    
    # Non-housing keywords (check first)
    non_housing = ['wireless', 'antenna', 'hot tub', 'spa', 'fence', 'retaining wall',
                   'pergola', 'arbor', 'deck only', 'carport only', 'parking only',
                   'alcohol', 'beer', 'wine', 'spirits', 'liquor', 'restaurant',
                   'karaoke', 'karoke', 'cannabis', 'vehicle wash', 'car wash',
                   'boiler', 'hvac', 'gate', 'driveway only', 'warehouse use',
                   'martial arts', 'fitness', 'medical practitioner', 'vocational',
                   'poultry', 'vehicle parts', 'sign', 'catering',
                   'shake shack', 'raising cane']
    
    is_clearly_non_housing = any(kw in desc for kw in non_housing)
    
    # Housing production detection
    housing_indicators = [
        'dwelling unit', 'residential unit', 'mixed-use', 'mixed use',
        'new single family', 'new sfr', 'construct a new',
        'residential building', 'residential development',
        'housing development', 'student housing',
        'group living', 'live/work', 'live-work',
        'duplex', 'triplex',
    ]
    
    has_housing_indicator = any(kw in desc for kw in housing_indicators)
    
    # Classify
    if is_clearly_non_housing and not has_housing_indicator and max_units == 0:
        result['housing_type'] = 'Non-Housing'
        result['is_housing_production'] = False
    elif max_units >= 5 or result['has_density_bonus']:
        result['housing_type'] = 'Major Housing (5+ units)'
        result['is_housing_production'] = True
        result['estimated_units'] = max_units
    elif result['has_adu']:
        result['housing_type'] = 'ADU'
        result['is_housing_production'] = True
        result['estimated_units'] = max(1, max_units)
    elif result['is_conversion'] and ('residential' in desc or 'dwelling' in desc or 'living' in desc):
        result['housing_type'] = 'Conversion to Housing'
        result['is_housing_production'] = True
        result['estimated_units'] = max(1, max_units)
    elif result['adds_bedrooms']:
        result['housing_type'] = 'Bedroom Addition'
        result['is_housing_production'] = True
        result['estimated_units'] = 0  # bedrooms, not units
    elif has_housing_indicator or max_units > 0:
        result['housing_type'] = 'Small Housing (1-4 units)'
        result['is_housing_production'] = True
        result['estimated_units'] = max(1, max_units)
    elif 'addition' in desc and ('residential' in desc or 'living' in desc):
        result['housing_type'] = 'Residential Addition'
        result['is_housing_production'] = False  # additions don't add units
    else:
        result['housing_type'] = 'Non-Housing'
        result['is_housing_production'] = False
    
    return pd.Series(result)

# Apply classification
classified = df.apply(classify_and_extract, axis=1)
df = pd.concat([df, classified], axis=1)

# Summary
print(f"📊 CLASSIFICATION RESULTS:\n")
housing = df[df['is_housing_production']]
non_housing = df[~df['is_housing_production']]

print(f"   🏗️ Housing Production:  {len(housing)} projects, ~{housing['estimated_units'].sum():,.0f} estimated units")
print(f"   ❌ Non-Housing:         {len(non_housing)} projects\n")

print(f"📋 By type:")
for htype, group in df.groupby('housing_type'):
    units = group['estimated_units'].sum()
    print(f"   {htype}: {len(group)} projects" + (f" (~{units:,.0f} units)" if units > 0 else ""))

print(f"\n📋 Density Bonus projects: {df['has_density_bonus'].sum()}")
print(f"📋 SB-330 projects: {df['is_sb330'].sum()}")
print(f"📋 ADU projects: {df['has_adu'].sum()}")
print(f"📋 Conversion projects: {df['is_conversion'].sum()}")

# Show housing production projects
print(f"\n{'='*70}")
print(f"\n🏗️ HOUSING PRODUCTION PROJECTS ({len(housing)}):\n")
for _, row in housing.sort_values('estimated_units', ascending=False).iterrows():
    bonus = ' 🏛️DB' if row['has_density_bonus'] else ''
    sb = ' 📋SB330' if row['is_sb330'] else ''
    adu = ' 🏠ADU' if row['has_adu'] else ''
    print(f"   {row['Permit_Number']:>15}: {row['estimated_units']:>4.0f} units | {row['housing_type']}{bonus}{sb}{adu}")
    print(f"{'':>18} {row['Address']} | {row['Record_Status']}")
    print(f"{'':>18} {row['Description'][:80]}")
    print()

# Save
df.to_sql('active_zoning_classified', conn, if_exists='replace', index=False)
conn.close()
print(f"💾 Saved 'active_zoning_classified' to accela_reports.db")

🏠 CLASSIFYING PERMITS & EXTRACTING UNIT COUNTS

📊 CLASSIFICATION RESULTS:

   🏗️ Housing Production:  57 projects, ~3,102 estimated units
   ❌ Non-Housing:         96 projects

📋 By type:
   ADU: 4 projects (~4 units)
   Bedroom Addition: 5 projects
   Conversion to Housing: 4 projects (~9 units)
   Major Housing (5+ units): 32 projects (~3,075 units)
   Non-Housing: 91 projects
   Residential Addition: 5 projects
   Small Housing (1-4 units): 12 projects (~14 units)

📋 Density Bonus projects: 25
📋 SB-330 projects: 12
📋 ADU projects: 6
📋 Conversion projects: 15


🏗️ HOUSING PRODUCTION PROJECTS (57):

       ZP2024-0058:  276 units | Major Housing (5+ units) 📋SB330
                   2700 SHATTUCK Ave | In Review
                   SB330 project to demolish an existing commercial structure and construct a eight

       ZP2020-0104:  257 units | Major Housing (5+ units) 🏛️DB
                   1914 FIFTH St | Under Review
                   demolish an existing parking lot and portions o

In [7]:
# CELL: Start building the planner lookup table
import pandas as pd
import sqlite3

DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'
conn = sqlite3.connect(DB_PATH)

# Start the planner table with what we know
planners = [
    {
        'permit_number': 'ZP2024-0058',
        'address': '2700 SHATTUCK Ave',
        'planner_name': 'Sharon Gong',
        'planner_title': 'Senior Planner',
        'planner_phone': '(510) 981-7429',
        'planner_email': 'sgong@cityofberkeley.info',
        'source_document': '2024-10-10_Complete Letter_2700 Shattuck.pdf',
    },
]

df_planners = pd.DataFrame(planners)
df_planners.to_sql('project_planners', conn, if_exists='replace', index=False)
conn.close()

print("💾 Started project_planners table with 1 record")
print("   ZP2024-0058 → Sharon Gong, Senior Planner")
print("\n   We'll populate this for all 57 housing projects by downloading")
print("   their Complete/Incomplete letters from Accela attachments.")

💾 Started project_planners table with 1 record
   ZP2024-0058 → Sharon Gong, Senior Planner

   We'll populate this for all 57 housing projects by downloading
   their Complete/Incomplete letters from Accela attachments.


## Friday start to build APR

In [8]:
# CELL: Morning check — verify both databases
import sqlite3, os, pandas as pd
from datetime import datetime

print(f"☀️ Good morning! {datetime.now().strftime('%A, %B %d, %Y %H:%M')}\n")
print("="*70)

for db_name, db_path in [
    ('berkeley.db', '/Users/johngage/berkeley-data/databases/berkeley.db'),
    ('accela_reports.db', '/Users/johngage/berkeley-data/databases/accela_reports.db'),
]:
    if os.path.exists(db_path):
        size = os.path.getsize(db_path) / 1024 / 1024
        conn = sqlite3.connect(db_path)
        tables = conn.execute(
            "SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name"
        ).fetchall()
        print(f"\n📁 {db_name} ({size:.1f} MB):")
        for name, ttype in tables:
            count = conn.execute(f"SELECT COUNT(*) FROM [{name}]").fetchone()[0]
            icon = '👁️' if ttype == 'view' else '📋'
            print(f"   {icon} {name}: {count:,} rows")
        conn.close()
    else:
        print(f"\n❌ {db_name}: NOT FOUND at {db_path}")

print(f"\n{'='*70}")
print("✅ Ready to continue")

☀️ Good morning! Friday, March 20, 2026 09:41


📁 berkeley.db (50.0 MB):
   📋 addresses_arcgis: 65,459 rows
   📋 corridor_boundaries: 3 rows
   📋 corridor_far: 332 rows
   📋 corridor_ownership: 332 rows
   📋 development_potential: 41 rows
   📋 licenses: 13,004 rows
   📋 licenses_fts: 13,004 rows
   📋 licenses_fts_config: 1 rows
   📋 licenses_fts_data: 189 rows
   📋 licenses_fts_docsize: 12,970 rows
   📋 licenses_fts_idx: 170 rows
   📋 parcel_zones: 29,024 rows
   📋 parcels: 29,024 rows
   📋 parcels_arcgis: 29,024 rows
   📋 rent_control: 1,098 rows
   📋 zoning_districts: 42 rows
   📋 zoning_projects_with_parcels: 154 rows
   👁️ corridor_master: 430 rows
   👁️ parcels_addresses_joined: 65,297 rows
   👁️ parcels_full: 65,507 rows

📁 accela_reports.db (0.2 MB):
   📋 active_landuse_v1_2_ActiveLandUse_V1: 135 rows
   📋 active_landuse_v1_ActiveLandUse_V1: 154 rows
   📋 active_landuse_v1_all_ActiveLandUse_V1: 155 rows
   📋 active_zoning_classified: 153 rows
   📋 active_zoning_projects: 153 row

In [9]:
# CELL: Verify the master view has zoning data
import sqlite3, pandas as pd

conn = sqlite3.connect('/Users/johngage/berkeley-data/databases/berkeley.db')

# Check parcels_full has zone data
sample = pd.read_sql("""
    SELECT apn_norm, SitusAddre, zone_class, zone_description, 
           middle_housing_eligible, density_bonus_eligible, UseCode
    FROM parcels_full 
    WHERE zone_class IS NOT NULL
    LIMIT 10
""", conn)

print(f"✅ parcels_full with zoning — sample:\n")
display(sample)

# Summary stats
stats = pd.read_sql("""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN zone_class IS NOT NULL THEN 1 ELSE 0 END) as zoned,
        SUM(CASE WHEN middle_housing_eligible = 1 THEN 1 ELSE 0 END) as middle_housing,
        SUM(CASE WHEN density_bonus_eligible = 1 THEN 1 ELSE 0 END) as density_bonus
    FROM parcels_full
""", conn)
print(f"\n📊 Master view stats:")
print(f"   Total rows:              {stats.iloc[0]['total']:,}")
print(f"   With zoning:             {stats.iloc[0]['zoned']:,}")
print(f"   Middle housing eligible: {stats.iloc[0]['middle_housing']:,}")
print(f"   Density bonus eligible:  {stats.iloc[0]['density_bonus']:,}")

conn.close()

✅ parcels_full with zoning — sample:



,apn_norm,SitusAddre,zone_class,zone_description,middle_housing_eligible,density_bonus_eligible,UseCode
0,016142800202,3208 SHATTUCK AVE BERKELEY 94705,C-SA,South Area Commercial — mixed use,0,1,2400
1,016142505700,6618 SHATTUCK AVE BERKELEY 94609,C-SA,South Area Commercial — mixed use,0,1,8100
2,016142202200,2320 WOOLSEY ST BERKELEY 94705,C-C,Corridor Commercial — housing above ground floor,0,1,9300
3,016142202000,6699 TELEGRAPH AVE BERKELEY 94609,C-C,Corridor Commercial — housing above ground floor,0,1,9300
4,016142202000,6699 TELEGRAPH AVE BERKELEY 94609,C-C,Corridor Commercial — housing above ground floor,0,1,9300
5,016142202000,6699 TELEGRAPH AVE BERKELEY 94609,C-C,Corridor Commercial — housing above ground floor,0,1,9300
6,016142202000,6699 TELEGRAPH AVE BERKELEY 94609,C-C,Corridor Commercial — housing above ground floor,0,1,9300
7,016142202000,6699 TELEGRAPH AVE BERKELEY 94609,C-C,Corridor Commercial — housing above ground floor,0,1,9300
8,016142202400,2314 WOOLSEY ST BERKELEY 94705,C-C,Corridor Commercial — housing above ground floor,0,1,9300
9,016142202600,2312 WOOLSEY ST BERKELEY 94705,R-2,Multi-Unit 2 — up to 8 units by right + ADUs,1,1,1100



📊 Master view stats:
   Total rows:              65,507
   With zoning:             65,370
   Middle housing eligible: 30,151
   Density bonus eligible:  64,472


In [10]:
# CELL: Create enriched schema tables
import sqlite3

DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'
conn = sqlite3.connect(DB_PATH)

# 1. Project Documents — every attachment for each housing project
conn.execute("""
    CREATE TABLE IF NOT EXISTS project_documents (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        permit_number TEXT NOT NULL,
        address TEXT,
        document_date TEXT,
        document_name TEXT,
        document_type TEXT,
        file_size_kb REAL,
        upload_date TEXT,
        last_updated TEXT,
        planner_extracted TEXT,
        planner_email_extracted TEXT,
        notes TEXT
    )
""")

# 2. Project Planners — multiple planners per project over time
conn.execute("DROP TABLE IF EXISTS project_planners")
conn.execute("""
    CREATE TABLE project_planners (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        permit_number TEXT NOT NULL,
        address TEXT,
        planner_name TEXT NOT NULL,
        planner_title TEXT,
        planner_phone TEXT,
        planner_email TEXT,
        source_document TEXT,
        date_first_seen TEXT,
        date_last_seen TEXT,
        is_current INTEGER DEFAULT 1
    )
""")

# Re-insert Sharon Gong
conn.execute("""
    INSERT INTO project_planners 
    (permit_number, address, planner_name, planner_title, planner_phone, 
     planner_email, source_document, date_first_seen, date_last_seen, is_current)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", ('ZP2024-0058', '2700 SHATTUCK Ave', 'Sharon Gong', 'Senior Planner',
      '(510) 981-7429', 'sgong@cityofberkeley.info',
      '2024-10-10_Complete Letter_2700 Shattuck.pdf',
      '2024-10-10', '2026-02-27', 1))

# 3. Permit Pipeline — full lifecycle tracking for APR
conn.execute("""
    CREATE TABLE IF NOT EXISTS permit_pipeline (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        permit_number TEXT NOT NULL,
        address TEXT,
        apn_norm TEXT,
        project_description TEXT,
        housing_type TEXT,
        estimated_units INTEGER,
        affordable_vli INTEGER DEFAULT 0,
        affordable_low INTEGER DEFAULT 0,
        affordable_moderate INTEGER DEFAULT 0,
        above_moderate INTEGER DEFAULT 0,
        has_density_bonus INTEGER DEFAULT 0,
        streamlining_provision TEXT,
        applicant_name TEXT,
        applicant_company TEXT,
        owner_name TEXT,
        current_planner TEXT,
        date_application_submitted TEXT,
        date_deemed_complete TEXT,
        date_entitled TEXT,
        date_building_permit_issued TEXT,
        date_building_permit_finaled TEXT,
        date_certificate_of_occupancy TEXT,
        current_status TEXT,
        pipeline_stage TEXT,
        is_stalled INTEGER DEFAULT 0,
        stall_reason TEXT,
        notes TEXT
    )
""")

# 4. Owner enrichment
conn.execute("""
    CREATE TABLE IF NOT EXISTS owner_enrichment (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        owner_name TEXT NOT NULL,
        apn_norm TEXT,
        address TEXT,
        source TEXT,
        owner_type TEXT,
        other_properties_count INTEGER,
        other_jurisdictions TEXT,
        mailing_address TEXT,
        mailing_address_shared_count INTEGER,
        notes TEXT
    )
""")

# Insert the Regrid data for 2700 Shattuck
conn.execute("""
    INSERT INTO owner_enrichment
    (owner_name, address, source, owner_type, other_properties_count,
     mailing_address, mailing_address_shared_count, notes)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", ('2700 SHATTUCK LLC', '2700 SHATTUCK AVE', 'SF Chronicle/Regrid',
      'LLC (single-purpose entity)', 0,
      '1600 SHATTUCK AVE BERKELEY CA 94709', 6,
      'Mailing address shared with 6 other CA properties - likely developer office'))

conn.commit()

# Verify
tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()
print("📊 Tables in accela_reports.db:\n")
for t in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM [{t[0]}]").fetchone()[0]
    print(f"   📋 {t[0]}: {count} rows")

conn.close()

print("\n✅ Schema created with initial data:")
print("   • project_documents — ready for attachment scraping")
print("   • project_planners — 1 record (Sharon Gong)")
print("   • permit_pipeline — ready for lifecycle tracking")
print("   • owner_enrichment — 1 record (2700 Shattuck LLC)")

📊 Tables in accela_reports.db:

   📋 active_landuse_v1_2_ActiveLandUse_V1: 135 rows
   📋 active_landuse_v1_ActiveLandUse_V1: 154 rows
   📋 active_landuse_v1_all_ActiveLandUse_V1: 155 rows
   📋 active_zoning_classified: 153 rows
   📋 active_zoning_projects: 153 rows
   📋 owner_enrichment: 1 rows
   📋 permit_pipeline: 0 rows
   📋 project_documents: 0 rows
   📋 project_planners: 1 rows
   📋 sqlite_sequence: 2 rows

✅ Schema created with initial data:
   • project_documents — ready for attachment scraping
   • project_planners — 1 record (Sharon Gong)
   • permit_pipeline — ready for lifecycle tracking
   • owner_enrichment — 1 record (2700 Shattuck LLC)


In [11]:
# CELL: Save scraped Accela record details to database
import sqlite3, json

DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'
conn = sqlite3.connect(DB_PATH)

# Create record_details table
conn.execute("""
    CREATE TABLE IF NOT EXISTS record_details (
        permit_number TEXT PRIMARY KEY,
        record_status TEXT,
        address TEXT,
        description TEXT,
        applicant_name TEXT,
        applicant_company TEXT,
        applicant_email TEXT,
        owner_name TEXT,
        attachment_count INTEGER,
        planner_name TEXT,
        planner_email TEXT,
        scraped_date TEXT DEFAULT (date('now'))
    )
""")

# Insert what we scraped today
records = [
    ('ZP2024-0058', 'In Review', '2700 SHATTUCK Ave', 
     'SB330 project, 8-story mixed-use, 276 dwelling units',
     'Isaiah Stackhouse', 'Trachtenberg Architects', 'isaiah@trachtenbergarch.com',
     '2700 SHATTUCK LLC', 30, 'Sharon Gong', 'sgong@cityofberkeley.info'),
    ('ZP2024-0181', 'Pending Final Action', '2029 UNIVERSITY Ave',
     '23-story, 240-unit housing, 15% VLI, 15% moderate, 100% density bonus',
     'Steve Heaton', 'Laconia Development', None,
     'TALAI MOHAMMAD E & KOKAB S TRS', 9, None, None),
    ('ZP2020-0104', 'Under Review', '1914 FIFTH St',
     '7-story mixed-use, 257 dwelling units, 26 VLI, density bonus',
     'Isiah Stackhouse', 'Trachtenberg Architects', 'isaiah@trachtenbergarch.com',
     'JAMESTOWN PREMIER BERKELEY GRO', 10, None, None),
    ('ZP2021-0158', 'In Review', '130 BERKELEY Sq',
     '50 unit multi-family mixed-use',
     'Christie Deng', 'Studio KDA', 'christie@studiokda.com',
     'BERKELEY STATION PARTNERS LLC', None, None, None),
]

for r in records:
    conn.execute("""INSERT OR REPLACE INTO record_details 
        (permit_number, record_status, address, description,
         applicant_name, applicant_company, applicant_email,
         owner_name, attachment_count, planner_name, planner_email)
        VALUES (?,?,?,?,?,?,?,?,?,?,?)""", r)

conn.commit()
count = conn.execute("SELECT COUNT(*) FROM record_details").fetchone()[0]
print(f"✅ Saved {count} record details to accela_reports.db")

# Show what we have
import pandas as pd
df = pd.read_sql("SELECT permit_number, address, record_status, applicant_company, owner_name FROM record_details", conn)
print(f"\n{df.to_string(index=False)}")

# Show remaining permits to scrape
remaining = [
    'ZP2022-0046', 'ZP2022-0116', 'ZP2022-0132', 'ZP2022-0149', 
    'ZP2022-0171', 'ZP2023-0058', 'ZP2023-0070', 'ZP2023-0089',
    'ZP2023-0090', 'ZP2023-0095', 'ZP2023-0096', 'ZP2023-0099',
    'ZP2023-0107', 'ZP2024-0027', 'ZP2024-0033', 'ZP2024-0047',
    'ZP2024-0067', 'ZP2024-0071', 'ZP2024-0074', 'ZP2024-0076',
    'ZP2024-0077', 'ZP2024-0079', 'ZP2024-0114', 'ZP2024-0126',
    'ZP2024-0131', 'ZP2024-0162', 'ZP2024-0182', 'ZP2025-0101',
    'ZP2025-0105', 'ZP2025-0107', 'ZP2026-0006', 'ZP2026-0007',
    'ZP2026-0015',
]
print(f"\n⏳ {len(remaining)} major housing permits still to scrape")
conn.close()

✅ Saved 4 record details to accela_reports.db

permit_number             address        record_status       applicant_company                     owner_name
  ZP2024-0058   2700 SHATTUCK Ave            In Review Trachtenberg Architects              2700 SHATTUCK LLC
  ZP2024-0181 2029 UNIVERSITY Ave Pending Final Action     Laconia Development TALAI MOHAMMAD E & KOKAB S TRS
  ZP2020-0104       1914 FIFTH St         Under Review Trachtenberg Architects JAMESTOWN PREMIER BERKELEY GRO
  ZP2021-0158     130 BERKELEY Sq            In Review              Studio KDA  BERKELEY STATION PARTNERS LLC

⏳ 33 major housing permits still to scrape


In [12]:
# CELL: Save all 19 scraped Accela records
import sqlite3

DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'
conn = sqlite3.connect(DB_PATH)

conn.execute("""CREATE TABLE IF NOT EXISTS record_details (
    permit_number TEXT PRIMARY KEY, record_status TEXT, address TEXT,
    description TEXT, applicant_name TEXT, applicant_company TEXT,
    applicant_email TEXT, owner_name TEXT, attachment_count INTEGER,
    planner_name TEXT, planner_email TEXT, scraped_date TEXT DEFAULT (date('now'))
)""")

records = [
  ('ZP2024-0058','In Review','2700 SHATTUCK Ave','SB330, 8-story, 276 dwelling units','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2700 SHATTUCK LLC',30,'Sharon Gong','sgong@cityofberkeley.info'),
  ('ZP2024-0181','Pending Final Action','2029 UNIVERSITY Ave','23-story, 240-unit, 15% VLI, 15% moderate, 100% density bonus','Steve Heaton','Laconia Development',None,'TALAI MOHAMMAD E & KOKAB S TRS',9,None,None),
  ('ZP2020-0104','Under Review','1914 FIFTH St','7-story, 257 units, 26 VLI, density bonus','Isiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','JAMESTOWN PREMIER BERKELEY GRO',10,None,None),
  ('ZP2021-0158','In Review','130 BERKELEY Sq','50 unit multi-family mixed-use','Christie Deng','Studio KDA','christie@studiokda.com','BERKELEY STATION PARTNERS LLC',None,None,None),
  ('ZP2022-0046','In Review','3000 SHATTUCK Ave','10-story, 166 units, 17 VLI, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','3000 SHATTUCK AVENUE LLC',None,None,None),
  ('ZP2022-0116','In Review','2920 SHATTUCK Ave','10-story, 221 units, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2900 SHATTUCK AVENUE LLC',None,None,None),
  ('ZP2022-0132','Incomplete Pending Applicant','2847 SHATTUCK Ave','9-story, 112 units, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','GOLDENBERG RUTH L TR',None,None,None),
  ('ZP2022-0149','In Review','2420 SHATTUCK Ave','17-story, 132 units, density bonus','Isiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2420 SHATTUCK LLC',None,None,None),
  ('ZP2022-0171','In Review','2601 SAN PABLO Ave','8-story, 223 units, SB 330, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2601 SPA LLC',None,None,None),
  ('ZP2023-0058','Under Review','2720 SAN PABLO Ave','8-story, 84 units, 9 VLI, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2720 SPA LLC',None,None,None),
  ('ZP2023-0070','Under Review','1790 UNIVERSITY Ave','5-story, 17 units, 2 VLI, SB 330, density bonus','Kathy Wang',None,None,'1776 UNIVERSITY AVENUE LLC',None,None,None),
  ('ZP2023-0089','Incomplete Pending Applicant','2441 LE CONTE Ave','5-story, 65 GLA units, 7 VLI, density bonus','Erik Waterman',None,None,'UCB LECONTE LLC',None,None,None),
  ('ZP2023-0090','In Review','2733 SAN PABLO Ave','8-story mixed use residential, density bonus','ISAIAH STACKHOUSE','Trachtenberg Architects','isaiah@trachtenbergarch.com','MCGEE ROBERT D & LOIS J TRS',None,None,None),
  ('ZP2023-0095','Incomplete Pending Applicant','2660 BANCROFT Way','8-story, 78 studios, 8 VLI, density bonus','GENEVA HESNER',None,None,'AMI LLC',None,None,None),
  ('ZP2023-0099','In Review','2109 MILVIA St','14-story, 105 units, SB-330, density bonus','ISAIAH STACKHOUSE','Trachtenberg Architects','isaiah@trachtenbergarch.com','CENTURY PROPERTIES LLC',None,None,None),
  ('ZP2024-0027','Corrections Pending Applicant','2614 TELEGRAPH Ave','5-story, 32 units, 3 VLI + 2 Low, SB-330, density bonus','LEILA GHAZ',None,None,'AVENUE T PROPERTY LLC',None,None,None),
  ('ZP2024-0047','Corrections Pending Applicant','2450 SHATTUCK Ave','8-story, 94 units, SB-330','Bill Schrader',None,None,'GORDON JOHN K & MITCHELL JANIS',None,None,None),
  ('ZP2024-0067','Pending Final Action','2276 SHATTUCK Ave','Morse Block, 84 units (56 base + 28 bonus), 9 VLI, 50% density bonus','Jane Eisberg',None,None,'PASAND COURTYARD LLC',None,None,None),
  ('ZP2024-0071','Corrections Pending Applicant','2955 SHATTUCK Ave','8-story, 74 units, 4 affordable, density bonus','ERIK WATERMAN',None,None,'VICTORIA ASSOCIATES NO.2',None,None,None),
  ('ZP2024-0074','Corrections Pending Applicant','1581 UNIVERSITY Ave','7-story, 158 units, 14 VLI + 9 moderate, 82.5% density bonus, SB-330','ISAIAH STACKHOUSE','Trachtenberg Architects','isaiah@trachtenbergarch.com','HON MANAGEMENT INC',None,None,None),
  ('ZP2024-0076','In Review','2720 SAN PABLO Ave','8-story, 117 units, 10 VLI + 6 moderate, 80% density bonus, SB-330','ISAIAH STACKHOUSE','Trachtenberg Architects','isaiah@trachtenbergarch.com','2720 SPA LLC',None,None,None),
  ('ZP2024-0077','In Review','2847 SHATTUCK Ave','9-story, 136 units, 14% VLI, 46.25% density bonus, SB-330','ISAIAH STACKHOUSE','Trachtenberg Architects','isaiah@trachtenbergarch.com','GOLDENBERG RUTH L TR',None,None,None),
  ('ZP2024-0079','In Review','2036 BANCROFT Way','8-story, 87 units, 4 VLI, SB-330, landmarked buildings demo','2322 SHATTUCK AVE LLC',None,None,'2322 SHATTUCK AVENUE LLC',None,None,None),
]

for r in records:
    conn.execute("""INSERT OR REPLACE INTO record_details 
        (permit_number, record_status, address, description,
         applicant_name, applicant_company, applicant_email,
         owner_name, attachment_count, planner_name, planner_email)
        VALUES (?,?,?,?,?,?,?,?,?,?,?)""", r)

conn.commit()
count = conn.execute("SELECT COUNT(*) FROM record_details").fetchone()[0]

# Quick stats
import pandas as pd
df = pd.read_sql("SELECT * FROM record_details", conn)
conn.close()

print(f"✅ Saved {count} record details to accela_reports.db\n")
print(f"📊 Status breakdown:")
for status, cnt in df['record_status'].value_counts().items():
    print(f"   {status}: {cnt}")

print(f"\n📊 Top applicants:")
for app, cnt in df['applicant_name'].value_counts().head(5).items():
    print(f"   {app}: {cnt} projects")

# Remaining to scrape
remaining = [
    'ZP2024-0114', 'ZP2024-0126', 'ZP2024-0131', 'ZP2024-0162', 
    'ZP2024-0182', 'ZP2023-0096', 'ZP2023-0107', 'ZP2024-0033',
    'ZP2025-0101', 'ZP2025-0105', 'ZP2025-0107',
    'ZP2026-0006', 'ZP2026-0007', 'ZP2026-0015',
]
print(f"\n⏳ {len(remaining)} major housing permits still to scrape")
print(f"   {', '.join(remaining)}")

✅ Saved 23 record details to accela_reports.db

📊 Status breakdown:
   In Review: 11
   Corrections Pending Applicant: 4
   Under Review: 3
   Incomplete Pending Applicant: 3
   Pending Final Action: 2

📊 Top applicants:
   Isaiah Stackhouse: 6 projects
   ISAIAH STACKHOUSE: 5 projects
   Isiah Stackhouse: 2 projects
   Steve Heaton: 1 projects
   Christie Deng: 1 projects

⏳ 14 major housing permits still to scrape
   ZP2024-0114, ZP2024-0126, ZP2024-0131, ZP2024-0162, ZP2024-0182, ZP2023-0096, ZP2023-0107, ZP2024-0033, ZP2025-0101, ZP2025-0105, ZP2025-0107, ZP2026-0006, ZP2026-0007, ZP2026-0015


In [14]:
# CELL: Save ALL scraped Accela records (27 complete + 8 remaining)
import sqlite3

DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'
conn = sqlite3.connect(DB_PATH)

conn.execute("DROP TABLE IF EXISTS record_details")
conn.execute("""CREATE TABLE record_details (
    permit_number TEXT PRIMARY KEY, record_status TEXT, address TEXT,
    description TEXT, applicant_name TEXT, applicant_company TEXT,
    applicant_email TEXT, owner_name TEXT, attachment_count INTEGER,
    planner_name TEXT, planner_email TEXT, scraped_date TEXT DEFAULT (date('now'))
)""")

records = [
  ('ZP2024-0058','In Review','2700 SHATTUCK Ave','SB330, 8-story, 276 dwelling units','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2700 SHATTUCK LLC',30,'Sharon Gong','sgong@cityofberkeley.info'),
  ('ZP2024-0181','Pending Final Action','2029 UNIVERSITY Ave','23-story, 240-unit, 15% VLI, 15% moderate, 100% density bonus','Steve Heaton','Laconia Development',None,'TALAI MOHAMMAD E & KOKAB S TRS',9,None,None),
  ('ZP2020-0104','Under Review','1914 FIFTH St','7-story, 257 units, 26 VLI, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','JAMESTOWN PREMIER BERKELEY GRO',10,None,None),
  ('ZP2021-0158','In Review','130 BERKELEY Sq','50 unit multi-family mixed-use','Christie Deng','Studio KDA','christie@studiokda.com','BERKELEY STATION PARTNERS LLC',None,None,None),
  ('ZP2022-0046','In Review','3000 SHATTUCK Ave','10-story, 166 units, 17 VLI, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','3000 SHATTUCK AVENUE LLC',None,None,None),
  ('ZP2022-0116','In Review','2920 SHATTUCK Ave','10-story, 221 units, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2900 SHATTUCK AVENUE LLC',None,None,None),
  ('ZP2022-0132','Incomplete Pending Applicant','2847 SHATTUCK Ave','9-story, 112 units, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','GOLDENBERG RUTH L TR',None,None,None),
  ('ZP2022-0149','In Review','2420 SHATTUCK Ave','17-story, 132 units, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2420 SHATTUCK LLC',None,None,None),
  ('ZP2022-0171','In Review','2601 SAN PABLO Ave','8-story, 223 units, SB 330, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2601 SPA LLC',None,None,None),
  ('ZP2023-0058','Under Review','2720 SAN PABLO Ave','8-story, 84 units, 9 VLI, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2720 SPA LLC',None,None,None),
  ('ZP2023-0070','Under Review','1790 UNIVERSITY Ave','5-story, 17 units, 2 VLI, SB 330, density bonus','Kathy Wang',None,None,'1776 UNIVERSITY AVENUE LLC',None,None,None),
  ('ZP2023-0089','Incomplete Pending Applicant','2441 LE CONTE Ave','5-story, 65 GLA units, 7 VLI, density bonus','Erik Waterman',None,None,'UCB LECONTE LLC',None,None,None),
  ('ZP2023-0090','In Review','2733 SAN PABLO Ave','8-story mixed use residential, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','MCGEE ROBERT D & LOIS J TRS',None,None,None),
  ('ZP2023-0095','Incomplete Pending Applicant','2660 BANCROFT Way','8-story, 78 studios, 8 VLI, density bonus','Geneva Hesner',None,None,'AMI LLC',None,None,None),
  ('ZP2023-0099','In Review','2109 MILVIA St','14-story, 105 units, SB-330, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','CENTURY PROPERTIES LLC',None,None,None),
  ('ZP2024-0027','Corrections Pending Applicant','2614 TELEGRAPH Ave','5-story, 32 units, 3 VLI + 2 Low, SB-330, density bonus','Leila Ghaz',None,None,'AVENUE T PROPERTY LLC',None,None,None),
  ('ZP2024-0047','Corrections Pending Applicant','2450 SHATTUCK Ave','8-story, 94 units, SB-330','Bill Schrader',None,None,'GORDON JOHN K & MITCHELL JANIS',None,None,None),
  ('ZP2024-0067','Pending Final Action','2276 SHATTUCK Ave','Morse Block, 84 units (56+28 bonus), 9 VLI, 50% density bonus','Jane Eisberg',None,None,'PASAND COURTYARD LLC',None,None,None),
  ('ZP2024-0071','Corrections Pending Applicant','2955 SHATTUCK Ave','8-story, 74 units, 4 affordable, density bonus','Erik Waterman',None,None,'VICTORIA ASSOCIATES NO.2',None,None,None),
  ('ZP2024-0074','Corrections Pending Applicant','1581 UNIVERSITY Ave','7-story, 158 units, 14 VLI + 9 moderate, 82.5% density bonus, SB-330','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','HON MANAGEMENT INC',None,None,None),
  ('ZP2024-0076','In Review','2720 SAN PABLO Ave','8-story, 117 units, 10 VLI + 6 moderate, 80% density bonus, SB-330','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2720 SPA LLC',None,None,None),
  ('ZP2024-0077','In Review','2847 SHATTUCK Ave','9-story, 136 units, 14% VLI, 46.25% density bonus, SB-330','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','GOLDENBERG RUTH L TR',None,None,None),
  ('ZP2024-0079','In Review','2036 BANCROFT Way','8-story, 87 units, 4 VLI, SB-330, landmarked demo','2322 SHATTUCK AVE LLC',None,None,'2322 SHATTUCK AVENUE LLC',None,None,None),
  ('ZP2024-0114','Approved','2138 KITTREDGE St','8-story, 66 units, 5 VLI, SB-330, density bonus','The Austin Group LLC',None,None,'CARLSON SUSAN TR',None,None,None),
  ('ZP2024-0126','Pending Final Action','2298 DURANT Ave','8-story, 65 units, 5 VLI, density bonus','Austin Springer',None,None,'2298 DURANT LP',None,None,None),
  ('ZP2024-0131','Incomplete Pending Applicant','2115 KITTREDGE St','23-story, 148 units + theater, 12 VLI + 12 moderate, density bonus','Christian Cerria',None,None,'AQUARIUS TWINS INC',None,None,None),
  ('ZP2024-0162','Pending Final Action','2425 DURANT Ave','20-story 200ft, 13 VLI + 13 moderate, zero parking, density bonus','Qian Wang',None,None,'2425 DURANT AVENUE LLC',None,None,None),
]

for r in records:
    conn.execute("""INSERT OR REPLACE INTO record_details 
        (permit_number, record_status, address, description,
         applicant_name, applicant_company, applicant_email,
         owner_name, attachment_count, planner_name, planner_email)
        VALUES (?,?,?,?,?,?,?,?,?,?,?)""", r)

conn.commit()

# Stats
import pandas as pd
df = pd.read_sql("SELECT * FROM record_details", conn)
conn.close()

print(f"✅ Saved {len(df)} record details to accela_reports.db\n")

# Extract unit counts from descriptions
import re
total_units = 0
for _, row in df.iterrows():
    d = row['description'] or ''
    matches = re.findall(r'(\d+)\s*(?:dwelling|unit|studio|GLA)', d, re.I)
    if matches:
        total_units += max(int(m) for m in matches)

print(f"🏗️ PIPELINE SUMMARY:")
print(f"   Projects scraped:     {len(df)}")
print(f"   Estimated total units: ~{total_units:,}")
print(f"\n📊 Status breakdown:")
for s, c in df['record_status'].value_counts().items():
    print(f"   {s}: {c}")

print(f"\n📊 Top applicants (normalized):")
df['applicant_norm'] = df['applicant_name'].str.title().str.replace(r'Isiah|Isaiah', 'Isaiah', regex=True)
for a, c in df['applicant_norm'].value_counts().head(5).items():
    print(f"   {a}: {c} projects")

# Remaining 8 to scrape next session
remaining = ['ZP2024-0182','ZP2023-0096','ZP2023-0107','ZP2024-0033',
             'ZP2025-0101','ZP2025-0105','ZP2025-0107',
             'ZP2026-0006','ZP2026-0007','ZP2026-0015']
print(f"\n⏳ {len(remaining)} permits still to scrape: {', '.join(remaining)}")

✅ Saved 27 record details to accela_reports.db

🏗️ PIPELINE SUMMARY:
   Projects scraped:     27
   Estimated total units: ~2,847

📊 Status breakdown:
   In Review: 11
   Pending Final Action: 4
   Incomplete Pending Applicant: 4
   Corrections Pending Applicant: 4
   Under Review: 3
   Approved: 1

📊 Top applicants (normalized):
   Isaiah Stackhouse: 13 projects
   Erik Waterman: 2 projects
   Steve Heaton: 1 projects
   Christie Deng: 1 projects
   Kathy Wang: 1 projects

⏳ 10 permits still to scrape: ZP2024-0182, ZP2023-0096, ZP2023-0107, ZP2024-0033, ZP2025-0101, ZP2025-0105, ZP2025-0107, ZP2026-0006, ZP2026-0007, ZP2026-0015


In [15]:
# CELL: Save ALL 34 scraped Accela records — COMPLETE
import sqlite3, re
import pandas as pd

DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'
conn = sqlite3.connect(DB_PATH)

conn.execute("DROP TABLE IF EXISTS record_details")
conn.execute("""CREATE TABLE record_details (
    permit_number TEXT PRIMARY KEY, record_status TEXT, address TEXT,
    description TEXT, applicant_name TEXT, applicant_company TEXT,
    applicant_email TEXT, owner_name TEXT, attachment_count INTEGER,
    planner_name TEXT, planner_email TEXT, scraped_date TEXT DEFAULT (date('now'))
)""")

records = [
  ('ZP2020-0104','Under Review','1914 FIFTH St','7-story, 257 units, 26 VLI, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','JAMESTOWN PREMIER BERKELEY GRO',10,None,None),
  ('ZP2021-0158','In Review','130 BERKELEY Sq','50 unit multi-family mixed-use','Christie Deng','Studio KDA','christie@studiokda.com','BERKELEY STATION PARTNERS LLC',None,None,None),
  ('ZP2022-0046','In Review','3000 SHATTUCK Ave','10-story, 166 units, 17 VLI, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','3000 SHATTUCK AVENUE LLC',None,None,None),
  ('ZP2022-0116','In Review','2920 SHATTUCK Ave','10-story, 221 units, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2900 SHATTUCK AVENUE LLC',None,None,None),
  ('ZP2022-0132','Incomplete Pending Applicant','2847 SHATTUCK Ave','9-story, 112 units, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','GOLDENBERG RUTH L TR',None,None,None),
  ('ZP2022-0149','In Review','2420 SHATTUCK Ave','17-story, 132 units, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2420 SHATTUCK LLC',None,None,None),
  ('ZP2022-0171','In Review','2601 SAN PABLO Ave','8-story, 223 units, SB 330, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2601 SPA LLC',None,None,None),
  ('ZP2023-0058','Under Review','2720 SAN PABLO Ave','8-story, 84 units, 9 VLI, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2720 SPA LLC',None,None,None),
  ('ZP2023-0070','Under Review','1790 UNIVERSITY Ave','5-story, 17 units, 2 VLI, SB 330, density bonus','Kathy Wang',None,None,'1776 UNIVERSITY AVENUE LLC',None,None,None),
  ('ZP2023-0089','Incomplete Pending Applicant','2441 LE CONTE Ave','5-story, 65 GLA units, 7 VLI, density bonus','Erik Waterman',None,None,'UCB LECONTE LLC',None,None,None),
  ('ZP2023-0090','In Review','2733 SAN PABLO Ave','8-story mixed use residential, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','MCGEE ROBERT D & LOIS J TRS',None,None,None),
  ('ZP2023-0095','Incomplete Pending Applicant','2660 BANCROFT Way','8-story, 78 studios, 8 VLI, density bonus','Geneva Hesner',None,None,'AMI LLC',None,None,None),
  ('ZP2023-0096','Incomplete Pending Applicant','2680 BANCROFT Way','Bancroft Hotel conversion, 15 dwellings + 22 GLA = 37 total, historic','Geneva Hesner',None,None,'AMI LLC',None,None,None),
  ('ZP2023-0099','In Review','2109 MILVIA St','14-story, 105 units, SB-330, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','CENTURY PROPERTIES LLC',None,None,None),
  ('ZP2023-0107','Approved','2462 BANCROFT Way','8-story, 66 units, 3 VLI, density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','ZENGS BERKELEY LLC',None,None,None),
  ('ZP2024-0027','Corrections Pending Applicant','2614 TELEGRAPH Ave','5-story, 32 units, 3 VLI + 2 Low, SB-330, density bonus','Leila Ghaz',None,None,'AVENUE T PROPERTY LLC',None,None,None),
  ('ZP2024-0033','Approved','2317 CHANNING Way','Mod 17→22 units, 5-story','Till Houtermans',None,None,'2317 CHANNING WAY LLC',None,None,None),
  ('ZP2024-0047','Corrections Pending Applicant','2450 SHATTUCK Ave','8-story, 94 units, SB-330','Bill Schrader',None,None,'GORDON JOHN K & MITCHELL JANIS',None,None,None),
  ('ZP2024-0058','In Review','2700 SHATTUCK Ave','SB330, 8-story, 276 dwelling units','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2700 SHATTUCK LLC',30,'Sharon Gong','sgong@cityofberkeley.info'),
  ('ZP2024-0067','Pending Final Action','2276 SHATTUCK Ave','Morse Block, 84 units (56+28 bonus), 9 VLI, 50% density bonus','Jane Eisberg',None,None,'PASAND COURTYARD LLC',None,None,None),
  ('ZP2024-0071','Corrections Pending Applicant','2955 SHATTUCK Ave','8-story, 74 units, 4 affordable, density bonus','Erik Waterman',None,None,'VICTORIA ASSOCIATES NO.2',None,None,None),
  ('ZP2024-0074','Corrections Pending Applicant','1581 UNIVERSITY Ave','7-story, 158 units, 14 VLI + 9 moderate, 82.5% density bonus, SB-330','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','HON MANAGEMENT INC',None,None,None),
  ('ZP2024-0076','In Review','2720 SAN PABLO Ave','8-story, 117 units, 10 VLI + 6 moderate, 80% density bonus, SB-330','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','2720 SPA LLC',None,None,None),
  ('ZP2024-0077','In Review','2847 SHATTUCK Ave','9-story, 136 units, 14% VLI, 46.25% density bonus, SB-330','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','GOLDENBERG RUTH L TR',None,None,None),
  ('ZP2024-0079','In Review','2036 BANCROFT Way','8-story, 87 units, 4 VLI, SB-330, landmarked demo','2322 SHATTUCK AVE LLC',None,None,'2322 SHATTUCK AVENUE LLC',None,None,None),
  ('ZP2024-0114','Approved','2138 KITTREDGE St','8-story, 66 units, 5 VLI, SB-330, density bonus','The Austin Group LLC',None,None,'CARLSON SUSAN TR',None,None,None),
  ('ZP2024-0126','Pending Final Action','2298 DURANT Ave','8-story, 65 units, 5 VLI, density bonus','Austin Springer',None,None,'2298 DURANT LP',None,None,None),
  ('ZP2024-0131','Incomplete Pending Applicant','2115 KITTREDGE St','23-story, 148 units + theater, 12 VLI + 12 moderate, density bonus','Christian Cerria',None,None,'AQUARIUS TWINS INC',None,None,None),
  ('ZP2024-0162','Pending Final Action','2425 DURANT Ave','20-story 200ft, 13 VLI + 13 moderate, zero parking, density bonus','Qian Wang',None,None,'2425 DURANT AVENUE LLC',None,None,None),
  ('ZP2024-0181','Pending Final Action','2029 UNIVERSITY Ave','23-story, 240-unit, 15% VLI, 15% moderate, 100% density bonus','Steve Heaton','Laconia Development',None,'TALAI MOHAMMAD E & KOKAB S TRS',9,None,None),
  ('ZP2024-0182','Pending Final Action','2029 UNIVERSITY Ave','23-story, 160 student units, 12 VLI + 12 moderate, 100% density bonus','Steve Heaton','Laconia Development',None,'TALAI MOHAMMAD E & KOKAB S TRS',None,None,None),
  ('ZP2025-0101','Under Review','2190 SHATTUCK Ave','Mod from 25→34 stories, density bonus increase','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','LMP SHATTUCK PROPERTY OWNER LLC',None,None,None),
  ('ZP2025-0105','Corrections Pending Applicant','2712 TELEGRAPH Ave','6-story, 50 units + 6 ADUs + 1 live/work = 57, density bonus','Austin Springer',None,None,'2712 TELEGRAPH LLC',None,None,None),
  ('ZP2025-0107','Corrections Pending Applicant','2001 CENTER St','Office→58 residential conversion, 6 VLI, AB-2097 no parking','Zhenyu Michael Song',None,None,'2001 CENTER STREET LLC',None,None,None),
  ('ZP2026-0006','Under Review','2138 KITTREDGE St','Mod from 66→73 units','William F. Schrader Jr.',None,None,'STUDIONE QOZB LLC',None,None,None),
]

for r in records:
    conn.execute("""INSERT OR REPLACE INTO record_details 
        (permit_number, record_status, address, description,
         applicant_name, applicant_company, applicant_email,
         owner_name, attachment_count, planner_name, planner_email)
        VALUES (?,?,?,?,?,?,?,?,?,?,?)""", r)

conn.commit()

# Analysis
df = pd.read_sql("SELECT * FROM record_details", conn)
conn.close()

# Extract unit counts
def extract_units(desc):
    if not desc: return 0
    matches = re.findall(r'(\d+)\s*(?:dwelling|unit|studio|GLA|student|residential)', desc, re.I)
    return max((int(m) for m in matches), default=0)

df['units'] = df['description'].apply(extract_units)

print(f"✅ Saved {len(df)} major housing project records\n")
print(f"🏗️ BERKELEY HOUSING PIPELINE SUMMARY")
print(f"{'='*60}")
print(f"   Total projects scraped:    {len(df)}")
print(f"   Estimated total units:     ~{df['units'].sum():,}")
print(f"\n📊 By status:")
for s, g in df.groupby('record_status'):
    print(f"   {s:35s} {len(g):>3} projects  ~{g['units'].sum():>5,} units")
print(f"\n📊 By applicant (normalized):")
df['app_norm'] = df['applicant_name'].str.title()
for a, c in df['app_norm'].value_counts().head(8).items():
    u = df[df['app_norm']==a]['units'].sum()
    print(f"   {a:30s} {c:>2} projects  ~{u:>5,} units")

print(f"\n📊 Largest projects:")
for _, r in df.nlargest(10, 'units').iterrows():
    print(f"   {r['permit_number']:>15}: {r['units']:>4} units | {r['address']:25s} | {r['record_status']}")

# Still need
print(f"\n⏳ Still need to scrape: ZP2026-0007, ZP2026-0015")

✅ Saved 35 major housing project records

🏗️ BERKELEY HOUSING PIPELINE SUMMARY
   Total projects scraped:    35
   Estimated total units:     ~3,298

📊 By status:
   Approved                              3 projects  ~  154 units
   Corrections Pending Applicant         6 projects  ~  466 units
   In Review                            11 projects  ~1,513 units
   Incomplete Pending Applicant          5 projects  ~  425 units
   Pending Final Action                  5 projects  ~  309 units
   Under Review                          5 projects  ~  431 units

📊 By applicant (normalized):
   Isaiah Stackhouse              15 projects  ~2,053 units
   Erik Waterman                   2 projects  ~  139 units
   Geneva Hesner                   2 projects  ~  100 units
   Steve Heaton                    2 projects  ~  160 units
   Austin Springer                 2 projects  ~  115 units
   The Austin Group Llc            1 projects  ~   66 units
   Zhenyu Michael Song             1 projects  ~   

In [18]:
# CELL: FINAL SUMMARY — fixed groupby
import sqlite3, re
import pandas as pd

DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'
conn = sqlite3.connect(DB_PATH)

# Add last 2 records
for r in [
  ('ZP2026-0007','In Review','2727 HASTE St','6-story, 45 units, 4 VLI + 3 low/moderate, 85% density bonus','Isaiah Stackhouse','Trachtenberg Architects','isaiah@trachtenbergarch.com','PYLKKANEN ANNA L & DARLENE',None,None,None),
  ('ZP2026-0015','In Review','2455 TELEGRAPH Ave','Amoeba Apts: 8-story, 68 units, 7 VLI, ground-floor record store','David Prinz',None,None,'GILLMAN SCOTT C TR & GILLMAN L',None,None,None),
]:
    conn.execute("""INSERT OR REPLACE INTO record_details 
        (permit_number, record_status, address, description,
         applicant_name, applicant_company, applicant_email,
         owner_name, attachment_count, planner_name, planner_email)
        VALUES (?,?,?,?,?,?,?,?,?,?,?)""", r)
conn.commit()

df = pd.read_sql("SELECT * FROM record_details", conn)
conn.close()

def extract_units(desc):
    if not desc: return 0
    matches = re.findall(r'(\d+)\s*(?:dwelling|unit|studio|GLA|student|residential)', desc, re.I)
    return max((int(m) for m in matches), default=0)

df['units'] = df['description'].apply(extract_units)

print(f"✅ ALL {len(df)} MAJOR HOUSING PROJECTS SCRAPED — 100% COMPLETE\n")
print(f"🏗️ BERKELEY HOUSING PIPELINE — FULL SUMMARY")
print(f"{'='*65}")
print(f"   Total projects:           {len(df)}")
print(f"   Estimated total units:    ~{df['units'].sum():,}")

print(f"\n📊 By status:")
for s, g in df.groupby('record_status', sort=False):
    print(f"   {s:35s} {len(g):>3} projects  ~{g['units'].sum():>5,} units")

print(f"\n📊 Approved / Pending Final (nearest to construction):")
near = df[df['record_status'].isin(['Approved','Pending Final Action'])]
for _, r in near.sort_values('units', ascending=False).iterrows():
    print(f"   {r['permit_number']:>15}: {r['units']:>4} units | {r['address']:25s} | {r['record_status']}")
print(f"   {'SUBTOTAL':>15}  {near['units'].sum():>4} units — nearest to building permits")

print(f"\n📊 Top 10 largest projects:")
for _, r in df.nlargest(10, 'units').iterrows():
    print(f"   {r['permit_number']:>15}: {r['units']:>4} units | {r['address']:25s} | {r['record_status']}")

print(f"\n📊 Multi-project applicants:")
df['app_norm'] = df['applicant_name'].str.title()
for a, g in df.groupby('app_norm', sort=False):
    if len(g) >= 2:
        print(f"   {a:30s} {len(g):>2} projects  ~{g['units'].sum():>5,} units")

print(f"\n✅ Database: accela_reports.db → record_details ({len(df)} rows)")
print(f"✅ Linked to: berkeley.db → parcels_full (29,024 parcels with zoning)")
print(f"\n🏗️ 37/37 major housing projects complete!")

✅ ALL 37 MAJOR HOUSING PROJECTS SCRAPED — 100% COMPLETE

🏗️ BERKELEY HOUSING PIPELINE — FULL SUMMARY
   Total projects:           37
   Estimated total units:    ~3,411

📊 By status:
   Under Review                          5 projects  ~  431 units
   In Review                            13 projects  ~1,626 units
   Incomplete Pending Applicant          5 projects  ~  425 units
   Approved                              3 projects  ~  154 units
   Corrections Pending Applicant         6 projects  ~  466 units
   Pending Final Action                  5 projects  ~  309 units

📊 Approved / Pending Final (nearest to construction):
       ZP2024-0182:  160 units | 2029 UNIVERSITY Ave       | Pending Final Action
       ZP2024-0067:   84 units | 2276 SHATTUCK Ave         | Pending Final Action
       ZP2023-0107:   66 units | 2462 BANCROFT Way         | Approved
       ZP2024-0114:   66 units | 2138 KITTREDGE St         | Approved
       ZP2024-0126:   65 units | 2298 DURANT Ave           | P

In [19]:
# CELL: Check BuildingEye and Berkeley Open Data for building permit datasets
import requests

# 1. Check Berkeley Open Data catalog for building permits
print("🔍 SEARCHING BERKELEY OPEN DATA FOR BUILDING PERMITS\n")
catalog_url = "https://data.cityofberkeley.info/api/catalog/v1"
for term in ['building permit', 'certificate of occupancy', 'housing permit', 'construction permit']:
    resp = requests.get(catalog_url, params={'q': term, 'limit': 5, 'only': 'datasets'}, timeout=10)
    if resp.status_code == 200:
        results = resp.json().get('results', [])
        for r in results:
            res = r.get('resource', {})
            name = res.get('name', '')
            did = res.get('id', '')
            desc = res.get('description', '')[:100]
            print(f"   📁 {name}")
            print(f"      ID: {did}")
            print(f"      {desc}")
            print()

# 2. Check BuildingEye
print("\n🔍 CHECKING BUILDINGEYE\n")
be_urls = [
    'https://buildingeye.com/berkeley',
    'https://www.buildingeye.com/berkeley',
    'https://berkeley.buildingeye.com',
]
for url in be_urls:
    try:
        resp = requests.get(url, timeout=10, allow_redirects=True)
        print(f"   {url} → {resp.status_code} (redirected to: {resp.url[:80]})")
    except Exception as e:
        print(f"   {url} → Error: {e}")

# 3. Try the GIS Building Safety layer to see year range
print("\n🔍 GIS BUILDING SAFETY LAYER — YEAR RANGE\n")
gis_url = "https://gis.cityofberkeley.info/arcgis/rest/services/Planning/Building_Safety/MapServer/3/query"
params = {
    'where': '1=1',
    'outStatistics': '[{"statisticType":"max","onStatisticField":"APPLICATIONYEAR","outStatisticFieldName":"max_year"},{"statisticType":"min","onStatisticField":"APPLICATIONYEAR","outStatisticFieldName":"min_year"},{"statisticType":"count","onStatisticField":"APPLICATIONYEAR","outStatisticFieldName":"total_count"}]',
    'f': 'json'
}
resp = requests.get(gis_url, params=params, timeout=15)
if resp.status_code == 200:
    data = resp.json()
    if 'features' in data and data['features']:
        stats = data['features'][0]['attributes']
        print(f"   Year range: {stats.get('min_year', '?')} to {stats.get('max_year', '?')}")
        print(f"   Total records: {stats.get('total_count', '?')}")
    else:
        print(f"   Response: {str(data)[:200]}")
else:
    print(f"   HTTP {resp.status_code}")

# 4. Also check the Land_Use_Planning service
print("\n🔍 GIS LAND USE PLANNING LAYERS\n")
lu_url = "https://gis.cityofberkeley.info/arcgis/rest/services/Planning/Land_Use_Planning/MapServer?f=json"
resp = requests.get(lu_url, timeout=10)
if resp.status_code == 200:
    data = resp.json()
    for layer in data.get('layers', []):
        print(f"   Layer {layer['id']}: {layer['name']}")

🔍 SEARCHING BERKELEY OPEN DATA FOR BUILDING PERMITS

   📁 Building Permits
      ID: ydr8-5enu
      <b>Note, 10/15/2025:</b> We have added a PERMIT_CONDITION column.

This dataset includes information

   📁 Building Permits
      ID: c2es-76ed
      Building permit applications made to The City of Calgary's Planning & Development department. 
For g

   📁 Building Permits
      ID: 76t5-zqzr
      All building permits issued or in progress within the city of Seattle.

   📁 Building Permits
      ID: vmzx-48vx
      Building Permits for the current and preceding 4 calendar years

   📁 General Building Permits
      ID: 24uj-dj8v
      List of issued building permits from the City of Edmonton - Urban Planning & Economy Department for 

   📁 DOB Certificate Of Occupancy
      ID: bs8b-p36w
      A Certificate of Occupancy (CO) states a building’s legal use and/or type of permitted occupancy. Ne

   📁 Certificate of Occupancy
      ID: 9npx-zf2y
      <div>Certificates of Occupancy issued 

In [20]:
# CELL: Fetch Berkeley Building Permits from Open Data (Socrata API)
# Dataset: ydr8-5enu — "Building Permits" with PERMIT_CONDITION column (added Oct 2025)
import requests
import pandas as pd
import sqlite3
import json

print("🏗️ FETCHING BERKELEY BUILDING PERMITS FROM OPEN DATA\n")
print("="*70)

BASE_URL = "https://data.cityofberkeley.info/resource/ydr8-5enu.json"

# First, get a small sample to see the columns
print("📋 Fetching sample to check fields...")
resp = requests.get(BASE_URL, params={'$limit': 5}, timeout=30)
print(f"   HTTP {resp.status_code}")

if resp.status_code == 200:
    sample = resp.json()
    if sample:
        print(f"   Fields: {list(sample[0].keys())}\n")
        print("   Sample record:")
        for k, v in sample[0].items():
            print(f"      {k}: {str(v)[:80]}")
        
        # Now fetch ALL 2024 records
        print(f"\n{'='*70}")
        print(f"\n📦 Fetching ALL 2024 building permits...")
        
        all_records = []
        offset = 0
        limit = 2000
        
        while True:
            params = {
                '$limit': limit,
                '$offset': offset,
                # Try filtering for 2024 — field name may vary
                '$where': "permit_number LIKE 'B2024%' OR permit_number LIKE 'B2025%' OR permit_number LIKE 'B2026%'",
            }
            resp = requests.get(BASE_URL, params=params, timeout=30)
            if resp.status_code != 200:
                print(f"   ❌ HTTP {resp.status_code} at offset {offset}")
                # Try without filter
                if offset == 0:
                    print("   Trying without filter...")
                    params = {'$limit': limit, '$offset': 0}
                    resp = requests.get(BASE_URL, params=params, timeout=30)
                    if resp.status_code == 200:
                        batch = resp.json()
                        all_records.extend(batch)
                        print(f"   Got {len(batch)} records without filter")
                break
            
            batch = resp.json()
            if not batch:
                break
            all_records.extend(batch)
            print(f"   Fetched {len(all_records):,} records...")
            offset += limit
            if len(batch) < limit:
                break
        
        df = pd.DataFrame(all_records)
        print(f"\n✅ Total records fetched: {len(df):,}")
        print(f"📋 Columns: {df.columns.tolist()}")
        
        if len(df) > 0:
            # Show column types and sample values
            print(f"\n📊 Column overview:")
            for col in df.columns:
                non_null = df[col].notna().sum()
                sample_val = df[col].dropna().iloc[0] if non_null > 0 else 'all null'
                print(f"   {col:30s} {non_null:>6} non-null | {str(sample_val)[:60]}")
            
            # Save to database
            DB_PATH = '/Users/johngage/berkeley-data/databases/accela_reports.db'
            conn = sqlite3.connect(DB_PATH)
            df.to_sql('building_permits_opendata', conn, if_exists='replace', index=False)
            conn.close()
            print(f"\n💾 Saved to accela_reports.db → building_permits_opendata ({len(df)} rows)")
    else:
        print("   Empty response")
else:
    print(f"   Error: {resp.text[:200]}")
    print("\n   Try fetching without the resource ID path:")
    # Alternative: use the dataset page URL approach
    alt_url = "https://data.cityofberkeley.info/api/views/ydr8-5enu/rows.json?accessType=DOWNLOAD"
    print(f"   Trying: {alt_url}")
    resp2 = requests.get(alt_url, timeout=60)
    print(f"   HTTP {resp2.status_code}")
    if resp2.status_code == 200:
        data = resp2.json()
        print(f"   Keys: {list(data.keys())[:10]}")

🏗️ FETCHING BERKELEY BUILDING PERMITS FROM OPEN DATA

📋 Fetching sample to check fields...
   HTTP 403
   Error: <!DOCTYPE html PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3c.org/TR/1999/REC-html401-19991224/loose.dtd"> <html> <head> <meta http-equiv="Content-Type" content="text/html; charset=UT

   Try fetching without the resource ID path:
   Trying: https://data.cityofberkeley.info/api/views/ydr8-5enu/rows.json?accessType=DOWNLOAD
   HTTP 403


In [26]:
# CELL: Fetch Berkeley Building Permits via SODA API
# Dataset ID: ydr8-5enu (confirmed Berkeley's building permits)
# Your Mac's IP should NOT be blocked by the WAF
import requests
import pandas as pd
import sqlite3

print("🏗️ BERKELEY BUILDING PERMITS VIA SODA API-yes\n")
print("="*70)

BASE = "https://data.cityofberkeley.info/resource/ydr8-5enu.json"

# Step 1: Test access with a tiny request
print("📋 Step 1: Testing access...")
resp = requests.get(BASE, params={'$limit': 3}, timeout=15)
print(f"   HTTP {resp.status_code}")

if resp.status_code == 403:
    print("\n   ❌ BLOCKED from Python too.")
    print("   Try getting a Socrata App Token:")
    print("   1. Go to https://data.cityofberkeley.info/profile/edit/developer_settings")
    print("   2. Create a new app token")
    print("   3. Re-run with the token below:\n")
    
    # Retry with app token if you have one
    APP_TOKEN = 'CyAwn3bXmniYiyJLNmnIxPT08'  # <-- paste your token here
    if APP_TOKEN:
        resp = requests.get(BASE, params={'$limit': 3, '$$app_token': APP_TOKEN}, timeout=15)
        print(f"   With new app token: HTTP {resp.status_code}")
    
    # Also try CSV export as fallback
    print("\n   Trying CSV export endpoint...")
    csv_url = "https://data.cityofberkeley.info/api/views/ydr8-5enu/rows.csv?accessType=DOWNLOAD"
    resp2 = requests.get(csv_url, timeout=60, stream=True)
    print(f"   CSV endpoint: HTTP {resp2.status_code}")
    if resp2.status_code == 200:
        import io
        df = pd.read_csv(io.StringIO(resp2.text))
        print(f"   ✅ GOT DATA VIA CSV! {len(df):,} rows")
    else:
        print("   ❌ CSV also blocked. Manual download needed.")
        print("   Open Safari → https://data.cityofberkeley.info")
        print("   Search for 'Building Permits' → Export → CSV")
        df = None
else:
    sample = resp.json()
    print(f"   ✅ Access works! {len(sample)} sample records")
    print(f"   Fields: {list(sample[0].keys())}")
    
    # Step 2: Count total records
    print(f"\n📊 Step 2: Counting records...")
    count_resp = requests.get(BASE, params={'$select': 'count(*)'}, timeout=15)
    if count_resp.status_code == 200:
        total = count_resp.json()[0].get('count', '?')
        print(f"   Total records: {total}")
    
    # Step 3: Fetch ALL records
    print(f"\n📦 Step 3: Fetching all records...")
    all_records = []
    offset = 0
    limit = 2000
    while True:
        r = requests.get(BASE, params={
            '$limit': limit, '$offset': offset, '$order': ':id'
        }, timeout=60)
        if r.status_code != 200:
            print(f"   ❌ HTTP {r.status_code} at offset {offset}")
            break
        batch = r.json()
        if not batch:
            break
        all_records.extend(batch)
        print(f"   {len(all_records):,} records...")
        offset += limit
        if len(batch) < limit:
            break
    
    df = pd.DataFrame(all_records)
    print(f"\n✅ Total: {len(df):,} records")

# Analysis if we have data
if df is not None and len(df) > 0:
    print(f"\n{'='*70}")
    print(f"📊 FIELD INVENTORY:\n")
    for col in sorted(df.columns):
        n = df[col].notna().sum()
        ex = df[col].dropna().iloc[0] if n > 0 else 'null'
        print(f"   {col:40s} {n:>6,} | {str(ex)[:50]}")
    
    # Save raw data
    DB = '/Users/johngage/berkeley-data/databases/accela_reports.db'
    conn = sqlite3.connect(DB)
    df.to_sql('building_permits_opendata', conn, if_exists='replace', index=False)
    conn.close()
    print(f"\n💾 Saved {len(df):,} rows → building_permits_opendata")

🏗️ BERKELEY BUILDING PERMITS VIA SODA API-yes

📋 Step 1: Testing access...
   HTTP 403

   ❌ BLOCKED from Python too.
   Try getting a Socrata App Token:
   1. Go to https://data.cityofberkeley.info/profile/edit/developer_settings
   2. Create a new app token
   3. Re-run with the token below:

   With new app token: HTTP 403

   Trying CSV export endpoint...
   CSV endpoint: HTTP 403
   ❌ CSV also blocked. Manual download needed.
   Open Safari → https://data.cityofberkeley.info
   Search for 'Building Permits' → Export → CSV


In [27]:
# CELL: Diagnose Berkeley Open Data WAF block
import requests
import json

print("🔍 DIAGNOSING BERKELEY OPEN DATA WAF BLOCK\n")
print("="*70)

# 1. What's our IP?
print("1. YOUR IP ADDRESS:")
try:
    ip_resp = requests.get("https://api.ipify.org?format=json", timeout=5)
    print(f"   {ip_resp.json()['ip']}")
except:
    print("   Could not determine")

# 2. Can we reach the homepage?
print("\n2. HOMEPAGE (no API):")
resp = requests.get("https://data.cityofberkeley.info", timeout=10)
print(f"   HTTP {resp.status_code} | Content-Length: {len(resp.text)}")

# 3. Try the catalog API (this worked before)
print("\n3. CATALOG API (worked earlier in session):")
resp = requests.get("https://data.cityofberkeley.info/api/catalog/v1",
                     params={'q': 'building', 'limit': 1}, timeout=10)
print(f"   HTTP {resp.status_code}")
if resp.status_code == 200:
    print(f"   ✅ Catalog works! Response: {resp.text[:150]}")

# 4. Try the data API with different headers
print("\n4. DATA API — HEADER EXPERIMENTS:")

tests = [
    ("No headers", {}),
    ("With User-Agent", {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) Safari/605.1.15"}),
    ("With Accept JSON", {"Accept": "application/json", "User-Agent": "Mozilla/5.0"}),
    ("With Referer", {"User-Agent": "Mozilla/5.0", "Referer": "https://data.cityofberkeley.info/"}),
    ("With App Token header", {"X-App-Token": "CyAwn3bXmniYiyJLNmnIxPT08", "User-Agent": "Mozilla/5.0"}),
    ("Socrata format", {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
        "Accept": "application/json",
        "X-App-Token": "CyAwn3bXmniYiyJLNmnIxPT08"
    }),
]

for label, headers in tests:
    try:
        r = requests.get("https://data.cityofberkeley.info/resource/ydr8-5enu.json",
                         params={'$limit': 1}, headers=headers, timeout=10)
        blocked = '403' if r.status_code == 403 else ''
        preview = r.text[:80] if r.status_code == 200 else ''
        print(f"   {label:30s} → HTTP {r.status_code} {blocked} {preview}")
    except Exception as e:
        print(f"   {label:30s} → Error: {e}")

# 5. Try alternative Socrata endpoints
print("\n5. ALTERNATIVE ENDPOINTS:")

alt_urls = [
    ("Metadata", f"https://data.cityofberkeley.info/api/views/ydr8-5enu.json"),
    ("Rows CSV", f"https://data.cityofberkeley.info/api/views/ydr8-5enu/rows.csv?accessType=DOWNLOAD"),
    ("Rows JSON", f"https://data.cityofberkeley.info/api/views/ydr8-5enu/rows.json?accessType=DOWNLOAD"),
    ("OData", f"https://data.cityofberkeley.info/OData.svc/ydr8-5enu"),
    ("GeoJSON", f"https://data.cityofberkeley.info/resource/ydr8-5enu.geojson?$limit=1"),
]

for label, url in alt_urls:
    try:
        r = requests.get(url, headers={
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
            "X-App-Token": "CyAwn3bXmniYiyJLNmnIxPT08"
        }, timeout=15, allow_redirects=True)
        size = len(r.text)
        preview = r.text[:100].replace('\n', ' ') if r.status_code == 200 else ''
        print(f"   {label:15s} → HTTP {r.status_code} ({size:,} bytes) {preview}")
    except Exception as e:
        print(f"   {label:15s} → Error: {e}")

# 6. Try a DIFFERENT Berkeley dataset to see if it's dataset-specific
print("\n6. OTHER BERKELEY DATASETS (is it just ydr8-5enu that's blocked?):")
other_datasets = [
    ("Business Licenses", "rwnf-bu3w"),
    ("311 Requests", "k489-uv4i"),
]
for label, did in other_datasets:
    try:
        r = requests.get(f"https://data.cityofberkeley.info/resource/{did}.json",
                         params={'$limit': 1},
                         headers={"User-Agent": "Mozilla/5.0", "X-App-Token": "CyAwn3bXmniYiyJLNmnIxPT08"},
                         timeout=10)
        print(f"   {label:25s} ({did}) → HTTP {r.status_code}")
        if r.status_code == 200:
            print(f"      Fields: {list(r.json()[0].keys())[:8]}...")
    except Exception as e:
        print(f"   {label:25s} → Error: {e}")

print(f"\n{'='*70}")
print("DIAGNOSIS COMPLETE")

🔍 DIAGNOSING BERKELEY OPEN DATA WAF BLOCK

1. YOUR IP ADDRESS:
   162.233.200.44

2. HOMEPAGE (no API):
   HTTP 200 | Content-Length: 826532

3. CATALOG API (worked earlier in session):
   HTTP 200
   ✅ Catalog works! Response: {
  "results" :
    [
      {
        "resource" :
          {
            "name" : "Building Permits",
            "id" : "c2es-76ed",
            "r

4. DATA API — HEADER EXPERIMENTS:
   No headers                     → HTTP 403 403 
   With User-Agent                → HTTP 403 403 
   With Accept JSON               → HTTP 403 403 
   With Referer                   → HTTP 403 403 
   With App Token header          → HTTP 403 403 
   Socrata format                 → HTTP 403 403 

5. ALTERNATIVE ENDPOINTS:
   Metadata        → HTTP 403 (3,689 bytes) 
   Rows CSV        → HTTP 403 (3,693 bytes) 
   Rows JSON       → HTTP 403 (3,694 bytes) 
   OData           → HTTP 500 (71 bytes) 
   GeoJSON         → HTTP 403 (3,691 bytes) 

6. OTHER BERKELEY DATASETS (is it ju

In [28]:
# CELL: Check your REAL IP vs what Python uses
import requests
import subprocess

# What IP does Python use?
r = requests.get("https://api.ipify.org?format=text", timeout=5)
print(f"Python requests IP:  {r.text}")

# What IP does your Mac use natively?
result = subprocess.run(["curl", "-s", "https://api.ipify.org"], capture_output=True, text=True)
print(f"curl IP:             {result.stdout}")

# Are they the same?
if r.text.strip() == result.stdout.strip():
    print("\n⚠️  Same IP — your traffic is ALL going through a proxy/VPN")
    print("   Check: Are you running a VPN? (Tailscale, Cloudflare WARP, etc.)")
    print("   Try: Disconnect VPN and re-run this cell")
else:
    print(f"\n✅ Different IPs! Use curl or set your proxy to bypass:")
    print(f"   Try setting: export no_proxy=data.cityofberkeley.info")

Python requests IP:  162.233.200.44
curl IP:             162.233.200.44

⚠️  Same IP — your traffic is ALL going through a proxy/VPN
   Check: Are you running a VPN? (Tailscale, Cloudflare WARP, etc.)
   Try: Disconnect VPN and re-run this cell


In [29]:
# CELL: Find what's proxying your traffic
import subprocess

print("🔍 IDENTIFYING YOUR VPN/PROXY\n")

# Check for common VPN processes
vpns = ['warp-cli', 'cloudflared', 'tailscaled', 'tailscale', 
        'wireguard', 'openvpn', 'mullvad', 'nordvpn', 'expressvpn',
        'Tunnelblick', 'viscosity']

result = subprocess.run(["ps", "aux"], capture_output=True, text=True)
for vpn in vpns:
    if vpn.lower() in result.stdout.lower():
        lines = [l for l in result.stdout.split('\n') if vpn.lower() in l.lower()]
        print(f"   ⚠️ FOUND: {vpn}")
        for l in lines[:2]:
            print(f"      {l.strip()[:120]}")

# Check network interfaces for tunnel adapters
print("\n📡 NETWORK INTERFACES:")
ifconfig = subprocess.run(["ifconfig"], capture_output=True, text=True)
for line in ifconfig.stdout.split('\n'):
    if line and not line.startswith('\t') and not line.startswith(' '):
        print(f"   {line.strip()}")

# Check if Cloudflare WARP is running
print("\n☁️ CLOUDFLARE WARP CHECK:")
warp = subprocess.run(["warp-cli", "status"], capture_output=True, text=True)
if warp.returncode == 0:
    print(f"   {warp.stdout.strip()}")
else:
    print(f"   warp-cli not found or not running")

# Check system proxy settings
print("\n🌐 SYSTEM PROXY SETTINGS:")
proxy = subprocess.run(["networksetup", "-getwebproxy", "Wi-Fi"], capture_output=True, text=True)
print(f"   HTTP:  {proxy.stdout.strip()}")
proxy_s = subprocess.run(["networksetup", "-getsecurewebproxy", "Wi-Fi"], capture_output=True, text=True)
print(f"   HTTPS: {proxy_s.stdout.strip()}")
socks = subprocess.run(["networksetup", "-getsocksfirewallproxy", "Wi-Fi"], capture_output=True, text=True)
print(f"   SOCKS: {socks.stdout.strip()}")

# Check env variables
import os
print("\n🔧 PROXY ENV VARIABLES:")
for var in ['http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'no_proxy']:
    val = os.environ.get(var, '')
    if val:
        print(f"   {var}={val}")

if not any(os.environ.get(v) for v in ['http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY']):
    print("   (none set)")

🔍 IDENTIFYING YOUR VPN/PROXY


📡 NETWORK INTERFACES:
   lo0: flags=8049<UP,LOOPBACK,RUNNING,MULTICAST> mtu 16384
   gif0: flags=8010<POINTOPOINT,MULTICAST> mtu 1280
   stf0: flags=0<> mtu 1280
   anpi1: flags=8863<UP,BROADCAST,SMART,RUNNING,SIMPLEX,MULTICAST> mtu 1500
   anpi0: flags=8863<UP,BROADCAST,SMART,RUNNING,SIMPLEX,MULTICAST> mtu 1500
   en3: flags=8863<UP,BROADCAST,SMART,RUNNING,SIMPLEX,MULTICAST> mtu 1500
   en4: flags=8863<UP,BROADCAST,SMART,RUNNING,SIMPLEX,MULTICAST> mtu 1500
   en1: flags=8963<UP,BROADCAST,SMART,RUNNING,PROMISC,SIMPLEX,MULTICAST> mtu 1500
   en2: flags=8963<UP,BROADCAST,SMART,RUNNING,PROMISC,SIMPLEX,MULTICAST> mtu 1500
   bridge0: flags=8863<UP,BROADCAST,SMART,RUNNING,SIMPLEX,MULTICAST> mtu 1500
   ap1: flags=8863<UP,BROADCAST,SMART,RUNNING,SIMPLEX,MULTICAST> mtu 1500
   en0: flags=8863<UP,BROADCAST,SMART,RUNNING,SIMPLEX,MULTICAST> mtu 1500
   awdl0: flags=8863<UP,BROADCAST,SMART,RUNNING,SIMPLEX,MULTICAST> mtu 1500
   llw0: flags=8863<UP,BROADCAST,SMART,RU

FileNotFoundError: [Errno 2] No such file or directory: 'warp-cli'

In [31]:
# CELL: Identify the VPN tunnels
import subprocess

print("🔍 IDENTIFYING VPN TUNNELS\n")

# Check each utun interface
for i in range(5):
    r = subprocess.run(["ifconfig", f"utun{i}"], capture_output=True, text=True)
    if r.returncode == 0:
        lines = r.stdout.strip().split('\n')
        inet = [l for l in lines if 'inet' in l]
        print(f"   utun{i}: {inet[0].strip() if inet else 'no IP assigned'}")

# Check routing table for default gateway
print("\n📡 DEFAULT ROUTE:")
r = subprocess.run(["route", "-n", "get", "default"], capture_output=True, text=True)
for line in r.stdout.split('\n'):
    if 'interface' in line.lower() or 'gateway' in line.lower():
        print(f"   {line.strip()}")

# Check for VPN-related processes
print("\n🔍 VPN PROCESSES:")
r = subprocess.run(["ps", "aux"], capture_output=True, text=True)
vpn_keywords = ['tailscale', 'vpn', 'anyconnect', 'globalprotect', 'pangp', 
                'openvpn', 'wireguard', 'fortivpn', 'cisco', 'palo', 
                'zscaler', 'netskope', 'prisma']
found = False
for line in r.stdout.split('\n'):
    for kw in vpn_keywords:
        if kw.lower() in line.lower():
            # Trim the line for readability
            parts = line.split()
            proc = ' '.join(parts[10:]) if len(parts) > 10 else line
            print(f"   ⚠️  {proc[:100]}")
            found = True
            break

if not found:
    print("   No obvious VPN process names found.")
    print("   Checking ALL running daemons...")
    r2 = subprocess.run(["launchctl", "list"], capture_output=True, text=True)
    for line in r2.stdout.split('\n'):
        for kw in ['vpn', 'tunnel', 'tailscale', 'anyconnect', 'global', 'zscaler', 'netskope', 'palo']:
            if kw.lower() in line.lower():
                print(f"   ⚠️  {line.strip()}")

# Quick check: can we reach Berkeley differently by specifying interface?
print("\n🧪 TRYING TO BYPASS VPN:")
# Get the real Wi-Fi interface IP
r3 = subprocess.run(["ipconfig", "getifaddr", "en0"], capture_output=True, text=True)
real_ip = r3.stdout.strip()
print(f"   en0 (Wi-Fi) local IP: {real_ip}")

# Try curl with --interface to force en0
r4 = subprocess.run([
    "curl", "-s", "--interface", "en0", "--connect-timeout", "5",
    "https://api.ipify.org"
], capture_output=True, text=True)
print(f"   curl via en0 IP: {r4.stdout.strip() or 'failed'}")

🔍 IDENTIFYING VPN TUNNELS

   utun0: inet6 fe80::1b7a:1602:8524:8a7%utun0 prefixlen 64 scopeid 0xf
   utun1: inet6 fe80::d755:a9f5:6195:5d2d%utun1 prefixlen 64 scopeid 0x10
   utun2: inet6 fe80::4335:8745:cc1b:8c8e%utun2 prefixlen 64 scopeid 0x11
   utun3: inet6 fe80::ce81:b1c:bd2c:69e%utun3 prefixlen 64 scopeid 0x12
   utun4: inet6 fe80::17c8:d132:cb1f:2a00%utun4 prefixlen 64 scopeid 0x13

📡 DEFAULT ROUTE:
   gateway: 192.168.1.254
   interface: en0
   flags: <UP,GATEWAY,DONE,STATIC,PRCLONING,GLOBAL>

🔍 VPN PROCESSES:
   No obvious VPN process names found.
   Checking ALL running daemons...

🧪 TRYING TO BYPASS VPN:
   en0 (Wi-Fi) local IP: 192.168.1.66
   curl via en0 IP: 162.233.200.44


In [32]:
# CELL: Test new IP after switching to FiberMain
import requests

# What's our IP now?
r = requests.get("https://api.ipify.org?format=text", timeout=5)
print(f"New IP: {r.text}")

# Can we reach Berkeley Open Data?
r2 = requests.get("https://data.cityofberkeley.info/resource/ydr8-5enu.json",
                   params={'$limit': 1}, timeout=10)
print(f"Berkeley Open Data: HTTP {r2.status_code}")
if r2.status_code == 200:
    data = r2.json()
    print(f"✅ ACCESS WORKS!")
    print(f"Fields: {list(data[0].keys())}")

New IP: 162.233.200.44
Berkeley Open Data: HTTP 403


In [33]:
# CELL: Quick diagnostic — is it your IP or the dataset?
import requests

print("🔍 FINAL DIAGNOSTIC\n")

# Your IP
r = requests.get("https://api.ipify.org?format=text", timeout=5)
print(f"Your IP: {r.text}")

# Which Wi-Fi are you on?
import subprocess
wifi = subprocess.run(["networksetup", "-getairportnetwork", "en0"], 
                       capture_output=True, text=True)
print(f"Wi-Fi: {wifi.stdout.strip()}")

# Test a bunch of Berkeley datasets to see the pattern
print(f"\n📊 TESTING MULTIPLE BERKELEY DATASETS:")
datasets = {
    'Business Licenses':  'rwnf-bu3w',
    'Building Permits':   'ydr8-5enu',
    'Code Enforcement':   'jke2-9kv6',
    '311 Requests':       'k489-uv4i',
    'Parcel Data':        'bhxd-e6up',
    'Fire Incidents':     'k7h2-m3bn',
}

working = []
blocked = []
for name, did in datasets.items():
    try:
        r = requests.get(f"https://data.cityofberkeley.info/resource/{did}.json",
                         params={'$limit': 1}, timeout=10)
        status = f"✅ {r.status_code}" if r.status_code == 200 else f"❌ {r.status_code}"
        print(f"   {name:25s} ({did}) → {status}")
        if r.status_code == 200:
            working.append(did)
        else:
            blocked.append(did)
    except Exception as e:
        print(f"   {name:25s} → Error: {e}")

print(f"\n📊 RESULT: {len(working)} working, {len(blocked)} blocked")

if working:
    print(f"\n   Your IP is NOT blanket-blocked — specific datasets are restricted.")
    print(f"   Building Permits (ydr8-5enu) may require authentication or")
    print(f"   may have been set to 'private' by the city.")
    
    # Check if the dataset metadata is accessible
    print(f"\n🔍 CHECKING DATASET METADATA:")
    r = requests.get("https://data.cityofberkeley.info/api/catalog/v1",
                     params={'ids': 'ydr8-5enu'}, timeout=10)
    if r.status_code == 200:
        meta = r.json()
        results = meta.get('results', [])
        if results:
            res = results[0].get('resource', {})
            print(f"   Name: {res.get('name')}")
            print(f"   Type: {res.get('type')}")
            print(f"   Updated: {res.get('updatedAt', '?')}")
            perms = results[0].get('metadata', {}).get('permissions', {})
            print(f"   Permissions: {perms}")
            classification = results[0].get('classification', {})
            print(f"   Classification: {classification}")

🔍 FINAL DIAGNOSTIC

Your IP: 162.233.200.44
Wi-Fi: You are not associated with an AirPort network.

📊 TESTING MULTIPLE BERKELEY DATASETS:
   Business Licenses         (rwnf-bu3w) → ✅ 200
   Building Permits          (ydr8-5enu) → ❌ 403
   Code Enforcement          (jke2-9kv6) → ❌ 403
   311 Requests              (k489-uv4i) → ❌ 403
   Parcel Data               (bhxd-e6up) → ✅ 200
   Fire Incidents            (k7h2-m3bn) → ❌ 403

📊 RESULT: 2 working, 4 blocked

   Your IP is NOT blanket-blocked — specific datasets are restricted.
   Building Permits (ydr8-5enu) may require authentication or
   may have been set to 'private' by the city.

🔍 CHECKING DATASET METADATA:
   Name: Building Permits
   Type: dataset
   Updated: 2026-03-20T12:12:25.000Z
   Permissions: {}
   Classification: {'categories': ['economy', 'education', 'politics'], 'tags': [], 'domain_category': 'Buildings', 'domain_tags': ['buildings', 'permits'], 'domain_metadata': [{'key': 'Metadata_Frequency', 'value': 'Data are u

In [34]:
# CELL: Find ACTUAL Berkeley datasets (not other cities)
import requests

print("🔍 FINDING REAL BERKELEY DATASETS\n")

# Search Berkeley's catalog specifically
resp = requests.get("https://data.cityofberkeley.info/api/catalog/v1",
                     params={'limit': 50, 'only': 'datasets'}, timeout=15)

if resp.status_code == 200:
    results = resp.json().get('results', [])
    print(f"Found {len(results)} datasets in Berkeley catalog:\n")
    
    for r in results:
        res = r.get('resource', {})
        name = res.get('name', '?')
        did = res.get('id', '?')
        domain = r.get('metadata', {}).get('domain', '?')
        desc = res.get('description', '')[:80]
        
        # Only show Berkeley's own datasets
        if 'berkeley' in domain.lower() or 'cityofberkeley' in domain.lower():
            marker = '🏗️' if 'permit' in name.lower() or 'building' in name.lower() else '  '
            print(f"   {marker} {name}")
            print(f"      ID: {did} | Domain: {domain}")
            print(f"      {desc}")
            print()
    
    # Also specifically search for permit/building datasets
    print(f"\n{'='*70}")
    print("🔍 SEARCHING FOR PERMIT-RELATED DATASETS:\n")
    for term in ['permit', 'building', 'housing', 'construction', 'certificate']:
        resp2 = requests.get("https://data.cityofberkeley.info/api/catalog/v1",
                             params={'q': term, 'limit': 10, 'only': 'datasets',
                                     'domains': 'data.cityofberkeley.info'}, timeout=10)
        if resp2.status_code == 200:
            for r in resp2.json().get('results', []):
                res = r.get('resource', {})
                domain = r.get('metadata', {}).get('domain', '')
                if 'cityofberkeley' in domain:
                    print(f"   📁 {res.get('name')} ({res.get('id')})")
                    print(f"      {res.get('description', '')[:100]}")
                    print()

# Test access to any permit datasets we find
print(f"\n{'='*70}")
print("🧪 TESTING ACCESS TO BERKELEY-ONLY DATASETS:\n")
# These are confirmed Berkeley datasets
berkeley_known = {
    'Business Licenses': 'rwnf-bu3w',
    'Parcel Data': 'bhxd-e6up',
}

for name, did in berkeley_known.items():
    r = requests.get(f"https://data.cityofberkeley.info/resource/{did}.json",
                     params={'$limit': 1}, timeout=10)
    print(f"   {name} ({did}): HTTP {r.status_code}")
    if r.status_code == 200:
        print(f"      Fields: {list(r.json()[0].keys())[:6]}...")

🔍 FINDING REAL BERKELEY DATASETS

Found 50 datasets in Berkeley catalog:


🔍 SEARCHING FOR PERMIT-RELATED DATASETS:

   📁 BESO Building Energy Data and Compliance Status - 15,000 sqft and greater (5vy5-rwja)
      This dataset contains the compliance status and reported energy metrics of medium and large building

   📁 BESO Data for GIS Portal (9is9-7t5k)
      BESO data to upload to GIS Portal

   📁 Monthly Residential Energy Use (77ue-wzxv)
      Actual electricity and natural gas meter reads for homes that participated in ME2/California Energy 

   📁 Car Share Totals (tscg-detc)
      Total number of car share vehicles and pods in Berkeley

   📁 Census, Housing, 2012 American Community Survey (7a6r-v8mp)
      Selected Housing Characteristics (Table DP04) from the Census Bureau 2012 American Community Survey 

   📁 Census Data 2000 And 2010 (bnkq-f2ge)
      2000 and 2010 US Census Bureau data for Berkeley as featured on the Bay Area Census website, http://


🧪 TESTING ACCESS TO BER

In [35]:
# CELL: Query BuildingEye API from Python
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import sqlite3

print("🏗️ QUERYING BUILDINGEYE FOR 2024 BUILDING PERMITS\n")
print("="*70)

session = requests.Session()

# Step 1: Load the building permits page to get session cookies
print("📋 Step 1: Getting session...")
resp = session.get("https://berkeley.buildingeye.com/building", timeout=15)
print(f"   HTTP {resp.status_code}, cookies: {list(session.cookies.keys())}")

# Step 2: Submit filtered search via POST
print("\n📋 Step 2: Submitting filtered search (2024, Building Permits)...")
form_data = {
    'PermitSearch[permit_type][]': 'BP',  # Building Permit
    'PermitSearch[date_from]': '01/01/2024',
    'PermitSearch[date_to]': '12/31/2024',
}
resp2 = session.post("https://berkeley.buildingeye.com/building", 
                      data=form_data, timeout=30)
print(f"   HTTP {resp2.status_code}, length: {len(resp2.text):,}")

# Parse the response for permit data
soup = BeautifulSoup(resp2.text, 'html.parser')

# Look for permit markers/data in the HTML
permits_found = re.findall(r'B\d{4}-\d{4,}', resp2.text)
print(f"   Permit numbers found in HTML: {len(permits_found)}")
if permits_found:
    print(f"   Sample: {permits_found[:5]}")

# Step 3: Try the locations endpoint with map bounds covering all of Berkeley
print("\n📋 Step 3: Querying locations API for all of Berkeley...")
loc_data = {
    'latitude': '37.8716',
    'longitude': '-122.2727',
    'zoom': '13',
    'bounds[southWest][lat]': '37.845',
    'bounds[southWest][lng]': '-122.325',
    'bounds[northEast][lat]': '37.905',
    'bounds[northEast][lng]': '-122.230',
    'permit_type[]': 'BP',
    'date_from': '01/01/2024',
    'date_to': '12/31/2024',
}
resp3 = session.post("https://berkeley.buildingeye.com/building/locations.html",
                      data=loc_data, timeout=30)
print(f"   HTTP {resp3.status_code}, length: {len(resp3.text):,}")

# Parse locations for permit data
permits_in_locations = re.findall(r'B\d{4}-\d{4,}', resp3.text)
addresses = re.findall(r'\d+\s+[A-Z][A-Z]+\s+(?:St|Ave|Way|Blvd|Rd|Dr|Ct|Pl|Ter|Ln)', resp3.text)
statuses = re.findall(r'(?:Issued|Finaled|Complete|Expired|Active|Open|Closed)', resp3.text, re.I)

print(f"   Permits: {len(permits_in_locations)}")
print(f"   Addresses: {len(addresses)}")
print(f"   Statuses: {statuses[:10]}")

if permits_in_locations:
    print(f"\n   Sample permits: {permits_in_locations[:10]}")

# Save raw HTML for inspection
with open('/Users/johngage/berkeley-data/buildingeye_2024_raw.html', 'w') as f:
    f.write(resp3.text)
print(f"\n💾 Saved raw response to buildingeye_2024_raw.html")

# Step 4: Also try to get the detail page for a known permit
print("\n📋 Step 4: Testing permit detail page...")
detail_resp = session.get("https://berkeley.buildingeye.com/building/permit/B2024-01544",
                          timeout=15)
print(f"   HTTP {detail_resp.status_code}, length: {len(detail_resp.text):,}")
detail_permits = re.findall(r'B\d{4}-\d{4,}', detail_resp.text)
print(f"   Permits in detail: {detail_permits[:5]}")

# Parse any tabular data
soup2 = BeautifulSoup(detail_resp.text, 'html.parser')
tables = soup2.find_all('table')
print(f"   Tables found: {len(tables)}")
for t in tables[:3]:
    rows = t.find_all('tr')
    for row in rows[:5]:
        cells = [td.get_text(strip=True) for td in row.find_all(['td', 'th'])]
        if cells:
            print(f"      {cells}")

# Save detail page
with open('/Users/johngage/berkeley-data/buildingeye_detail_sample.html', 'w') as f:
    f.write(detail_resp.text)
print(f"\n💾 Saved detail page to buildingeye_detail_sample.html")

🏗️ QUERYING BUILDINGEYE FOR 2024 BUILDING PERMITS

📋 Step 1: Getting session...
   HTTP 200, cookies: ['PHPSESSID', 'AWSELB']

📋 Step 2: Submitting filtered search (2024, Building Permits)...
   HTTP 200, length: 45,998
   Permit numbers found in HTML: 0

📋 Step 3: Querying locations API for all of Berkeley...
   HTTP 500, length: 8,549
   Permits: 0
   Addresses: 0
   Statuses: []

💾 Saved raw response to buildingeye_2024_raw.html

📋 Step 4: Testing permit detail page...
   HTTP 404, length: 8,683
   Permits in detail: []
   Tables found: 0

💾 Saved detail page to buildingeye_detail_sample.html


In [36]:
# CELL: CONSOLIDATED STATUS — What we have vs what APR needs
import os

print("📊 CONSOLIDATED DATA INVENTORY\n")
print("="*70)

# Check all databases
dbs = {
    'berkeley.db (parcels + zoning)': '/Users/johngage/berkeley-data/databases/berkeley.db',
    'accela_reports.db (permits)': '/Users/johngage/berkeley-data/databases/accela_reports.db', 
    'berkeley_housing_analysis.db': '/Users/johngage/berkeley-data/databases/berkeley_housing_analysis.db',
}

import sqlite3
for name, path in dbs.items():
    if os.path.exists(path):
        conn = sqlite3.connect(path)
        tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
        print(f"\n✅ {name} ({os.path.getsize(path)/1024/1024:.1f} MB)")
        for t in tables:
            count = conn.execute(f"SELECT COUNT(*) FROM [{t[0]}]").fetchone()[0]
            print(f"   {t[0]:40s} {count:>8,} rows")
        conn.close()
    else:
        print(f"\n❌ {name} — NOT FOUND")

# Check CSV files
csvs = {
    'housing_projects_FINAL.csv': '/Users/johngage/berkeley-data/data/processed/housing_projects_FINAL.csv',
    'alameda_lookup_complete.csv': '/Users/johngage/berkeley-data/data/reference/alameda_lookup_complete.csv',
}
print(f"\n{'='*70}")
print(f"\n📄 CSV FILES:")
import pandas as pd
for name, path in csvs.items():
    if os.path.exists(path):
        df = pd.read_csv(path, nrows=5)
        size = os.path.getsize(path) / 1024
        full_df = pd.read_csv(path)
        print(f"\n   ✅ {name} ({size:.0f} KB, {len(full_df):,} rows)")
        print(f"      Columns: {df.columns.tolist()}")
    else:
        print(f"\n   ❌ {name} — NOT FOUND")

# Check Accela reports folder
reports_dir = '/Users/johngage/berkeley-data/zoning_reports'
if os.path.exists(reports_dir):
    files = os.listdir(reports_dir)
    print(f"\n{'='*70}")
    print(f"\n📁 ACCELA REPORTS ({len(files)} files):")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(reports_dir, f)) / 1024
        print(f"   {f} ({size:.0f} KB)")

print(f"\n{'='*70}")
print(f"\n🎯 APR GAPS (from your compliance checklist):")
print(f"   1. Income breakdown (VLI/LI/MOD/AM) — CRITICAL, partially in descriptions")
print(f"   2. Building permit dates (issued/finaled) — MISSING, need Accela scraping")
print(f"   3. Certificate of Occupancy dates — MISSING, need 'Finaled' status from Accela")
print(f"   4. Tenure (Owner/Renter) — DERIVABLE from project type")
print(f"   5. APN coverage — WAS 3/115, NOW 98%+ via parcels_arcgis join")
print(f"   6. Unit category (SFD/SFA/2-4/5+/ADU) — DERIVABLE from descriptions")

📊 CONSOLIDATED DATA INVENTORY


✅ berkeley.db (parcels + zoning) (50.0 MB)
   licenses_fts                               13,004 rows
   licenses_fts_data                             189 rows
   licenses_fts_idx                              170 rows
   licenses_fts_docsize                       12,970 rows
   licenses_fts_config                             1 rows
   parcels                                    29,024 rows
   corridor_ownership                            332 rows
   corridor_far                                  332 rows
   rent_control                                1,098 rows
   corridor_boundaries                             3 rows
   licenses                                   13,004 rows
   parcels_arcgis                             29,024 rows
   addresses_arcgis                           65,459 rows
   zoning_districts                               42 rows
   development_potential                          41 rows
   zoning_projects_with_parcels                  154 ro

In [37]:
# CELL: Inventory ALL databases and key files
import os
import sqlite3

base = '/Users/johngage/berkeley-data'

print("📁 ALL .db FILES:\n")
for root, dirs, files in os.walk(base):
    for f in sorted(files):
        if f.endswith('.db'):
            path = os.path.join(root, f)
            size = os.path.getsize(path) / 1024 / 1024
            rel = os.path.relpath(path, base)
            
            # Count tables
            try:
                conn = sqlite3.connect(path)
                tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
                table_info = []
                for t in tables:
                    count = conn.execute(f"SELECT COUNT(*) FROM [{t[0]}]").fetchone()[0]
                    table_info.append(f"{t[0]}({count:,})")
                conn.close()
                print(f"   {rel:50s} {size:>6.1f} MB  → {', '.join(table_info)}")
            except Exception as e:
                print(f"   {rel:50s} {size:>6.1f} MB  → Error: {e}")

print(f"\n\n📁 ALL .csv FILES in data/:\n")
for root, dirs, files in os.walk(os.path.join(base, 'data')):
    for f in sorted(files):
        if f.endswith('.csv'):
            path = os.path.join(root, f)
            size = os.path.getsize(path) / 1024
            rel = os.path.relpath(path, base)
            print(f"   {rel:60s} {size:>8.0f} KB")

print(f"\n\n📁 ALL .ipynb FILES:\n")
for root, dirs, files in os.walk(base):
    # Skip archive and checkpoint dirs
    if '.ipynb_checkpoints' in root or 'node_modules' in root:
        continue
    for f in sorted(files):
        if f.endswith('.ipynb'):
            path = os.path.join(root, f)
            size = os.path.getsize(path) / 1024
            rel = os.path.relpath(path, base)
            print(f"   {rel:60s} {size:>6.0f} KB")

📁 ALL .db FILES:

   berkeleyshops-audience/audience.db                    0.3 MB  → mailchimp_audience(1,610)
   berkeleyshops-audience/archive/audience_2026-03-12.db    0.3 MB  → mailchimp_audience(1,560)
   databases/accela_reports.db                           0.3 MB  → active_landuse_v1_ActiveLandUse_V1(154), active_landuse_v1_2_ActiveLandUse_V1(135), active_landuse_v1_all_ActiveLandUse_V1(155), active_zoning_projects(153), active_zoning_classified(153), project_documents(0), sqlite_sequence(2), project_planners(1), permit_pipeline(0), owner_enrichment(1), record_details(37)
   databases/berkeley.db                                50.0 MB  → licenses_fts(13,004), licenses_fts_data(189), licenses_fts_idx(170), licenses_fts_docsize(12,970), licenses_fts_config(1), parcels(29,024), corridor_ownership(332), corridor_far(332), rent_control(1,098), corridor_boundaries(3), licenses(13,004), parcels_arcgis(29,024), addresses_arcgis(65,459), zoning_districts(42), development_potential(41), z

In [38]:
# CELL: THE REAL INVENTORY — What's in each "projects" table?
import sqlite3
import pandas as pd

print("📊 COMPARING ALL 'PROJECTS' TABLES\n")
print("="*70)

project_dbs = {
    'housing_projects.db': ('housing_projects', 84),
    'berkeley_housing_map.db': ('projects', 84),
    'berkeley_housing_analysis.db': ('projects', 115),
    'berkeley_housing_apr.db': ('projects', 115),
    'berkeley_address_centric.db': ('projects', 156),
    'datasette-deploy/berkeley_housing_map.db': ('projects', 157),
}

base = '/Users/johngage/berkeley-data'

for db_name, (table, expected) in project_dbs.items():
    path = f"{base}/{db_name}" if '/' not in db_name else f"{base}/{db_name}"
    try:
        conn = sqlite3.connect(path)
        df = pd.read_sql(f"SELECT * FROM [{table}] LIMIT 1", conn)
        count = pd.read_sql(f"SELECT COUNT(*) as n FROM [{table}]", conn).iloc[0]['n']
        cols = df.columns.tolist()
        conn.close()
        
        # Check for key APR fields
        has_apn = 'apn' in [c.lower() for c in cols]
        has_tenure = 'tenure' in [c.lower() for c in cols]
        has_vli = any('vli' in c.lower() for c in cols)
        has_lat = any('lat' in c.lower() for c in cols)
        has_status = any('status' in c.lower() for c in cols)
        has_desc = any('desc' in c.lower() for c in cols)
        
        print(f"\n📋 {db_name} → {table} ({count} rows)")
        print(f"   Columns ({len(cols)}): {cols[:8]}{'...' if len(cols) > 8 else ''}")
        print(f"   APR fields: APN={'✅' if has_apn else '❌'} Tenure={'✅' if has_tenure else '❌'} VLI={'✅' if has_vli else '❌'} Status={'✅' if has_status else '❌'} Desc={'✅' if has_desc else '❌'} Coords={'✅' if has_lat else '❌'}")
    except Exception as e:
        print(f"\n❌ {db_name}: {e}")

# Also compare with the CSV
print(f"\n{'='*70}")
print(f"\n📋 housing_projects_FINAL.csv (THE CANONICAL SOURCE)")
df = pd.read_csv(f'{base}/data/processed/housing_projects_FINAL.csv')
print(f"   {len(df)} rows, {len(df.columns)} columns")
print(f"   Columns: {df.columns.tolist()}")
print(f"   APN filled: {df['apn'].notna().sum()}/{len(df)}")
print(f"   Units total: {df['net_units'].sum():.0f}")
print(f"   VLI extracted: {df['vli_units_extracted'].notna().sum()}/{len(df)}")

# And the NEW data from this session
print(f"\n{'='*70}")
print(f"\n📋 NEW DATA FROM THIS SESSION:")
conn = sqlite3.connect(f'{base}/databases/accela_reports.db')
rd = pd.read_sql("SELECT * FROM record_details", conn)
conn.close()
print(f"   record_details: {len(rd)} projects with applicant/owner/description")
print(f"   Permit numbers: {rd['permit_number'].tolist()[:10]}...")

conn = sqlite3.connect(f'{base}/databases/berkeley.db')
pz = pd.read_sql("SELECT COUNT(DISTINCT apn_norm) as n FROM parcel_zones", conn)
conn.close()
print(f"   parcel_zones: {pz.iloc[0]['n']:,} parcels with zoning assigned")

# How many of the 37 Accela records overlap with the 115 in FINAL?
existing_permits = set()
for p in df['permits'].dropna():
    for pp in str(p).split(','):
        existing_permits.add(pp.strip())

overlap = rd[rd['permit_number'].isin(existing_permits)]
new_only = rd[~rd['permit_number'].isin(existing_permits)]
print(f"\n📊 OVERLAP ANALYSIS:")
print(f"   37 Accela-scraped records vs 115 in FINAL.csv:")
print(f"   Already in FINAL: {len(overlap)}")
print(f"   NEW (not in FINAL): {len(new_only)}")
if len(new_only) > 0:
    for _, r in new_only.iterrows():
        print(f"      {r['permit_number']:>15}: {r['address']:25s} ({r['record_status']})")

📊 COMPARING ALL 'PROJECTS' TABLES


❌ housing_projects.db: Execution failed on sql 'SELECT * FROM [housing_projects] LIMIT 1': no such table: housing_projects

❌ berkeley_housing_map.db: Execution failed on sql 'SELECT * FROM [projects] LIMIT 1': no such table: projects

❌ berkeley_housing_analysis.db: Execution failed on sql 'SELECT * FROM [projects] LIMIT 1': no such table: projects

❌ berkeley_housing_apr.db: Execution failed on sql 'SELECT * FROM [projects] LIMIT 1': no such table: projects

❌ berkeley_address_centric.db: Execution failed on sql 'SELECT * FROM [projects] LIMIT 1': no such table: projects

📋 datasette-deploy/berkeley_housing_map.db → projects (157 rows)
   Columns (16): ['project_id', 'address_display', 'address_normalized', 'latitude', 'longitude', 'data_source', 'net_units', 'status']...
   APR fields: APN=❌ Tenure=❌ VLI=❌ Status=✅ Desc=❌ Coords=✅


📋 housing_projects_FINAL.csv (THE CANONICAL SOURCE)
   115 rows, 25 columns
   Columns: ['id', 'address_display', 'a

In [39]:
# CELL: Add 3 new Approved projects to housing_projects_FINAL.csv
import pandas as pd
import sqlite3

base = '/Users/johngage/berkeley-data'
df = pd.read_csv(f'{base}/data/processed/housing_projects_FINAL.csv')

print(f"📋 BEFORE: {len(df)} projects, {df['net_units'].sum():.0f} units\n")

# The 3 new projects found this session (all Approved)
new_projects = [
    {
        'id': df['id'].max() + 1,
        'address_display': '2462 BANCROFT Way',
        'apn': None,  # will fill from lookup
        'owner': 'ZENGS BERKELEY LLC',
        'net_units': 66,
        'new_units': 66,
        'old_units': 0,
        'year': 2023,
        'permits': 'ZP2023-0107',
        'description': '8-story, 66 units, 3 VLI, density bonus, 2462 Bancroft',
        'status': 'Approved',
        'num_permits': 1,
        'project_size_category': 'Very Large (51-100)',
        'unit_category': '5+',
        'tenure': 'Renter',
        'sb35_flag': False,
        'sb330_flag': False,
        'ab2011_flag': False,
        'density_bonus': True,
        'density_bonus_pct': None,
        'vli_units_extracted': 3,
    },
    {
        'id': df['id'].max() + 2,
        'address_display': '2317 CHANNING Way',
        'apn': None,
        'owner': '2317 CHANNING WAY LLC',
        'net_units': 22,
        'new_units': 22,
        'old_units': 0,
        'year': 2024,
        'permits': 'ZP2024-0033',
        'description': 'Modification 17→22 units, 5-story residential',
        'status': 'Approved',
        'num_permits': 1,
        'project_size_category': 'Large (21-50)',
        'unit_category': '5+',
        'tenure': 'Renter',
        'sb35_flag': False,
        'sb330_flag': False,
        'ab2011_flag': False,
        'density_bonus': False,
        'density_bonus_pct': None,
        'vli_units_extracted': 0,
    },
    {
        'id': df['id'].max() + 3,
        'address_display': '2138 KITTREDGE St',
        'apn': None,
        'owner': 'CARLSON SUSAN TR',
        'net_units': 66,
        'new_units': 66,
        'old_units': 0,
        'year': 2024,
        'permits': 'ZP2024-0114',
        'description': '8-story, 66 units, 5 VLI, SB-330, density bonus',
        'status': 'Approved',
        'num_permits': 1,
        'project_size_category': 'Very Large (51-100)',
        'unit_category': '5+',
        'tenure': 'Renter',
        'sb35_flag': False,
        'sb330_flag': True,
        'ab2011_flag': False,
        'density_bonus': True,
        'density_bonus_pct': None,
        'vli_units_extracted': 5,
    },
]

new_df = pd.DataFrame(new_projects)

# Fill APNs from alameda lookup
lookup = pd.read_csv(f'{base}/data/reference/alameda_lookup_complete.csv',
                      usecols=['normalized_address', 'APN'])
import re
def norm(addr):
    if pd.isna(addr): return ''
    a = str(addr).upper().strip()
    for old, new in {' AVENUE':' AV',' AVE':' AV',' STREET':' ST',' WAY':' WY'}.items():
        a = a.replace(old, new)
    return re.sub(r'\s+', ' ', a).strip()

lookup['_n'] = lookup['normalized_address'].apply(norm)
apn_dict = lookup.drop_duplicates('_n').set_index('_n')['APN'].to_dict()

for idx, row in new_df.iterrows():
    apn = apn_dict.get(norm(row['address_display']))
    if apn:
        new_df.loc[idx, 'apn'] = apn
        print(f"   ✅ {row['permits']}: APN = {apn}")
    else:
        print(f"   ❌ {row['permits']}: APN not found for {row['address_display']}")

# Merge
merged = pd.concat([df, new_df], ignore_index=True)

print(f"\n📋 AFTER: {len(merged)} projects, {merged['net_units'].sum():.0f} units")
print(f"   Added: {len(new_df)} new Approved projects (+{new_df['net_units'].sum()} units)")

# Save
import shutil
from datetime import datetime
backup = f"{base}/data/backups/housing_projects_FINAL_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
shutil.copy(f'{base}/data/processed/housing_projects_FINAL.csv', backup)
print(f"\n💾 Backup: {backup}")

merged.to_csv(f'{base}/data/processed/housing_projects_FINAL.csv', index=False)
print(f"💾 Saved: housing_projects_FINAL.csv ({len(merged)} rows)")

print(f"\n📊 STATUS SUMMARY:")
for s, c in merged['status'].value_counts().items():
    u = merged[merged['status']==s]['net_units'].sum()
    print(f"   {s:35s} {c:>3} projects  {u:>5.0f} units")

📋 BEFORE: 115 projects, 5470 units

   ✅ ZP2023-0107: APN = 055 187802000
   ✅ ZP2024-0033: APN = 055 188400600
   ✅ ZP2024-0114: APN = 057 202901500

📋 AFTER: 118 projects, 5624 units
   Added: 3 new Approved projects (+154 units)

💾 Backup: /Users/johngage/berkeley-data/data/backups/housing_projects_FINAL_20260320_195539.csv
💾 Saved: housing_projects_FINAL.csv (118 rows)

📊 STATUS SUMMARY:
   Under Review                         40 projects   1685 units
   Incomplete Pending Applicant         19 projects    493 units
   In Review                            17 projects   1718 units
   Corrections Pending Applicant        17 projects    438 units
   Pending Final Action                 12 projects    883 units
   Approved                              6 projects    288 units
   Pending                               3 projects     74 units
   Resubmittal Pending Staff             2 projects     45 units
   Resubmittal Pending Review            1 projects      0 units
   On Hold          

/var/folders/zr/1lcy71z97n33bq1zyg3vtbp80000gn/T/ipykernel_32191/1588558707.py:108: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = pd.concat([df, new_df], ignore_index=True)
